# LASSI-Verify: Emperical Verification

**Argo Gateway API** Jupyter Notebook version

+ ***Post LASSI-Judge version with data collection for pass@k and energy_reduction@k***
+ No translation
+ Source code scripts from:
https://github.com/zjin-lcf/HeCBench/tree/master/src

NVIDIA GPU power measurements: https://pypi.org/project/nvidia-ml-py/  (pynvml)

Added support for AMD GPU device measurements and HIP code (ROCm)

In [1]:
from lassi.gpu_detector import detect_gpu_vendors

# Detect available GPUs
vendors = detect_gpu_vendors()
if not vendors:
    print("❌ No GPUs detected! LASSI requires GPU hardware.")
    raise SystemExit("Pipeline stopped - no GPU hardware found")

detected_vendor = vendors[0]['vendor']
gpu_count = vendors[0]['count']
print(f"🔍 Detected: {detected_vendor.upper()} with {gpu_count} GPU(s)")

# Vendor-specific initialization
if detected_vendor == "nvidia":
    # NVIDIA-specific setup
    import pynvml
    pynvml.nvmlInit()
    print("NVIDIA Driver Version:", pynvml.nvmlSystemGetDriverVersion())
    pynvml.nvmlShutdown()
    
    # Show NVIDIA system info
    !nvidia-smi
    !nvcc --version
    
elif detected_vendor == "amd":
    # AMD-specific setup
    from lassi.amd_power import amd_report_gpu_memory
    
    # Show AMD system info
    !rocminfo | head -20
    !rocm-smi
    !hipcc --version
    
print(f"\n✅ {detected_vendor.upper()} GPU environment ready for LASSI!")

🔍 Detected: NVIDIA with 1 GPU(s)
NVIDIA Driver Version: 570.211.01
Sun Mar 15 17:03:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 5000 Ada Gene...    Off |   00000000:01:00.0 Off |                  Off |
| N/A   39C    P8              3W /   80W |      14MiB /  16376MiB |      0%      Default |
|                                         |                        |     

## Setup

In [2]:
import os
import json
import csv
import time
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

# Import LASSI modules
import lassi.llm_config
# import lassi.pipeline # The Pipeline script calls the other LASSI libraries not imported here.
import lassi.pipeline_v4
from lassi.format_colors import color
from lassi.prompt_dictionary import PromptDictionary
from lassi.monitor import start_monitor, log_figure

# Enable the autoreload extension
%load_ext autoreload
%reload_ext autoreload

# Reload all modules imported with %aimport every time before executing subsequent code.
%autoreload 1

# We need to reload the libraries while under development
%aimport lassi.format_colors
%aimport lassi.prompt_dictionary
%aimport lassi.llm_config
%aimport lassi.llm_methods
%aimport lassi.pipeline_v4
%aimport lassi.get_power
%aimport lassi.argostream_curl
%aimport lassi.code_manager
%aimport lassi.pipeline_methods
%aimport lassi.verify
%aimport lassi.monitor

In [3]:
# ── LASSI Monitor — live output dashboard ──────────────────────────────────────
# Starts a local web server and streams all pipeline print() output to a browser
# tab in real-time.  Run this cell once after the imports cell above.
# Then open the link below in a new browser tab and keep it visible.
#
# If matplotlib figures are produced (EE task), they appear automatically in a
# sidebar panel on the right side of the dashboard — no extra calls needed.
#
# The dashboard accumulates the full session history, so a late-opened tab
# will still see everything printed since start_monitor() was called.
#
# SSH remote access: forward port 5007 to your local machine, then open
#   http://localhost:5007  in your local browser.
start_monitor()   # default port 5007 — pass port=XXXX to change

'http://localhost:5007'

## Code Generation Experiment - Setup + Run

_____
***STEP 1/2: Input experimental configuration parameters in the following method.***

In [4]:
def experimental_setup():
    # Load config file with source application names and input parameters
    with open("source_apps.json", "r") as f:
        source_apps = json.load(f)
    
    """
    [1] ENTER APPLICATION FOLDER NAME FROM HeCBench
        >> Execution input parameters provided.
    
    *** USE FOR LASSI-Verfify experiment: ***
    -- colorwheel ["10000", "256", "500000"] ["10000", "8", "1"]  # Usage: <range> <size> <repeat>
     (this one may show a lot of failures)

    HeCBench apps:
    -- matrix-rotate ["30000", "100"]  ["10000", "1"]  # Usage: %s <matrix size> <repeat>
    -- jacobi [""] (none required)
    -- layout ["2500"] ["1"]
    -- atomicCost ["16", "10"] ["1"]
    -- segment-reduce ["16384", "100"] (a replacement for atomicCost, same "Data compression and reduction" category in HeCBench)
       --> (may be trivial for LASSI-Verify)
    -- dense-embedding ["100000", "128", "100"] ["10000", "8", "1"]
    -- pathfinder ["100000", "10000", "2500"] ["100000", "1000", "5"] ["10000", "1000", "1000"]  # Usage: <column length> <row length> <pyramid_height>
    -- bsearch ["8388608", "1000"] ["10000", "1"]  # Usage: <number of elements> <repeat>
    -- entropy ["10000", "1024", "1"] ["8192", "8192", "500"]   # Usage: <width> <height> <repeat>
    -- colorwheel ["10000", "256", "500000"] ["10000", "8", "1"]  # Usage: <range> <size> <repeat>
    -- randomAccess ["10"] ["1"]  # Usage: <repeat>
    -- lid-driven-cavity [""]
    -- murmurhash3 ["750000, 500"]
    -- chacha20 ["300000"]
    -- jaccard ["1024, 512, 1000"]
    -- keogh ["256", "22500000", "100"]
    -- floydwarshall ["2048", "100", "16"]
    -- extrema ["750"]
    -- threadfence ["100", "250000000"]
    -- all-pairs-distance ["10000"]
    -- marchingCubes ["100"]

    Altis apps:
    -- level1/bfs ["--size 3"]
    -- level2/raytracing [""]
    -- level2/mandelbrot ["--imageSize 1024 --iterations 100000"]

    Miniapps:
    -- miniMDock ["-lfile", "./input/7cpa/7cpa_ligand.pdbqt", "-nrun 500"]
    -- XSBench ["-m", "event", "-l", "1700000000"]
    
    """

    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    
    # Enter source application to process through LASSI
    source_collection = "HeCBench"  # "Miniapps" "HeCBench"
    source_app = "lid-driven-cavity"  # "miniMDock" jacobi
    make_script = False  # Used only for Miniapps
    build_script = False  # False (for HeCBench) Use for Altis only
    

    """
    Valid source code folder names (under source_codes):
    1. HeCBench
    2. altis/src/cuda
    3. miniMDock/bin/
    """
    source_folder = source_apps[source_collection]["source_folder"]  # "altis/src/cuda/level2"    #  "HeCBench" "altis/src/cuda/level1"
    app_folder_name = source_apps[source_collection][source_app]["app_folder"]
    lines_of_code = source_apps[source_collection][source_app]["code_lines"]
    if make_script:
        app_name = source_app  # app_folder_name
        file_name = "Main"  # kernels
    else:
        app_name = "mt_" + app_folder_name + "-cuda_main"
        file_name = ""

    
    """
    [2] SELECT THE CODE TRANSLATION MAPPING DIRECTION EXPERIMENT CONFIGURATION
    
    CUDA -- OMP -- HIP -- SYCL -- FORTRAN
    """   
    # These should be the same if *not* doing translation.
    source_lang = "CUDA"
    target_lang = "CUDA"

    verifyprompt = False  # True or False -- Print out LLM prompt in stdout to preview
    # --> recommend keeping this False: the new monitor log has a better way of presenting all the LLM prompts,
    #  so we don't need to review it before
    
    # TOGGLE USE CODE KNOWLEDGE CONTEXT FOR TARGET
    use_context_summaries_only = True  # True or False
    few_shot_learning = False  # True or False
    
    # LLM judge code analysis (pre-cursor to LASSI-Verify)
    run_code_analysis = False  # True or False
    
    # THESE WILL NO LONGER BE OPTIONS
    # test_code_first = True  # True or False
    use_context = False  # True or False -- NOT USED IN v2, v3
    use_self_prompting = False  # True or False -- NOT USED IN v2, v3
    baseline_oneshot_goal = True  # True or False -- NOT USED IN v2, v3
    
    # -----------------------------------------------------------------------
    # TASK MODE CONFIGURATION
    # Set these two flags together depending on the task:
    #
    #   ENERGY EFFICIENCY (EE) task:
    #     power_measure    = True   -- measure GPU power/energy during execution
    #     yes_power_feedback = True -- run LASSI self-improvement loop using energy as the optimization metric
    #
    #   RUNTIME PERFORMANCE task:
    #     power_measure    = False  -- skip power measurement, use wall-clock duration instead
    #     yes_power_feedback = True -- run LASSI self-improvement loop using runtime as the optimization metric
    #
    #   CODE TRANSLATION task (one-shot, no iterative improvement):
    #     power_measure    = False  -- no measurement needed
    #     yes_power_feedback = False -- skip feedback loop entirely (generate once, compile, execute, judge, done)
    #     NOTE: translation mode still has a known infinite-loop-on-success issue when yes_power_feedback=False
    # -----------------------------------------------------------------------
    power_measure = False  # True (EE task) | False (runtime or translation task)
    vary_llm_temp = True  # True or False

    # TOGGLE LASSI SELF-IMPROVEMENT FEEDBACK LOOP
    yes_power_feedback = True  # True (EE or runtime task) | False (translation task, one-shot only)

    # -----------------------------------------------------------------------
    # LASSI-VERIFY TIER 2 CONFIGURATION
    # run_tier2=True: insert passthrough observation blocks and validate empirical equivalence
    # tier2_tolerance: relative floating-point tolerance for passthrough comparison (default 1e-6)
    # -----------------------------------------------------------------------
    run_tier2 = True         # True to enable Tier 2 empirical equivalence checking
    tier2_tolerance = 1e-6    # Relative tolerance for passthrough block comparison

    # LASSI-VERIFY TIER 3 CONFIGURATION
    # run_tier3=True: generate metamorphic contract and run variant filter (requires run_tier2=True)
    # tier3_tolerance: relative tolerance for metamorphic relation checks (default 1e-5, slightly looser than Tier 2)
    # -----------------------------------------------------------------------
    run_tier3 = True        # True to enable Tier 3 metamorphic testing
    tier3_tolerance = 1e-5   # Relative tolerance for metamorphic relation comparison

    # -----------------------------------------------------------------------
    # LLM-AS-A-JUDGE CONFIGURATION
    # run_llm_judge=True: enable LLM functional equivalence judge (pre-Tier-1/2 stop-gap)
    # run_llm_judge=False: pure three-tier mode — Tier 1 + Tier 2 determine validity only
    # -----------------------------------------------------------------------
    run_llm_judge = True     # True to enable LLM judge; False for pure three-tier LASSI-Verify mode

    # ASSIGN CODE COMPILE COMMAND
    code_build_script = ""
    code_compiler = ""
    code_compiler_kwds = ""
    if source_lang == "CUDA":
        if build_script:
            code_build_script = "build_" + app_name
        elif make_script:
            if source_app == "miniMDock":
                code_compiler = "make"
                code_compiler_kwds = "API=CUDA DEVICE=GPU CARD=NVIDIA NUMWI=64"
            elif source_app == "XSBench":
                code_compiler = "cuda/make"
        else:
            code_compiler = "nvcc"
            code_compiler_kwds = "-std=c++14 -Xcompiler -Wall -arch=sm_89 -O3"
            # -arch=sm_70 for aiops01 GPU (Argonne)  # -arch=sm_80 for A100s
            # -arch=sm_60 for P100
            # -arch=sm_89 for RTX 5000
        if yes_power_feedback:
            context_knowledge_selection = "cuda_optimization_summary"
            context_fewshot_examples = "cuda_c_few_shot_examples"
        else:
            context_knowledge_selection = "cuda"
    elif source_lang == "OMP":
        # For A100 on Linux_x86_64 or Linux_aarch64
        # code_compiler = "g++" # "/opt/nvidia/hpc_sdk/Linux_aarch64/2024/compilers/bin/nvc++"  # 
        # code_compiler_kwds = "-fopenmp -O3" # "-Wall -O3 -Minfo -mp=gpu -gpu=cc80"  # -c
        # !! TODO - REVIEW: code_compiler_kwds = "-Wall -O3 -Minfo=accel -mp=gpu -gpu=cc80"  # "-fopenmp -O3 -c"
        # For RTX 5000 Ada on Linux x86_64:
        code_compiler = "/opt/nvidia/hpc_sdk/Linux_x86_64/25.7/compilers/bin/nvc++" # "g++"
        code_compiler_kwds = "-Wall -O3 -Minfo=accel -mp=gpu -gpu=cc89"  # "-fopenmp -O3"
        context_knowledge_selection = "omp"
    elif source_lang == "HIP":
        if build_script:
            code_build_script = "build_" + app_name
        elif make_script:
            if source_app == "miniMDock":
                code_compiler = "make"
                code_compiler_kwds = "API=HIP DEVICE=GPU CARD=AMD NUMWI=64 HIP_PATH=/opt/rocm-6.3.3"
            elif source_app == "XSBench":
                code_compiler = "hip/make"
            else:
                code_compiler_kwds = ""
        else:
            code_compiler = "hipcc"
            code_compiler_kwds = "-std=c++14 -Wall -O3 --amdgpu-target=gfx908"
        if yes_power_feedback:
            context_knowledge_selection = "hip_optimization_summary"
            context_fewshot_examples = "hip_c_few_shot_examples"
        else:
            context_knowledge_selection = "hip"  # TODO

    # TOGGLE EXECUTE PIPELINE
    yes_execute = True  # True / False
    if yes_execute:
        # Enter execution input parameters, if needed
        # A list of parameter string values, e.g., ["10000", "1
        # ****************************
        # ---> EXPERIMENT UPDATE FIELD - Now, automated:
        exe_input_parameters = source_apps[source_collection][source_app]["app_params"]
    else:
        exe_input_parameters = [""]

    
    """
    [3] SELECT THE LLM TO USE FOR THE CODE TRANSLATION EXPERIMENT
    """
    # SELECT A MODEL

    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    
    # -- FROM ARGO:
    # model = "gpt35"  # (max token = 4096)
    # model = "gpt35large"  # (max token = 16384)
    # model = "gpt4large"  # (max token = 32768)
    # model = "gpt4turbo"  # (max token = 128000)
    # model = "gpto4mini"  # (max token = 128000)
    # model_coder = "gpto4mini"  # (max token = 128000)
    # model = "gpt4olatest"  # (max token = 128000)
    # model_coder = "gpt4olatest"  # (max token = 128000)
    # model_coder = "gpt4o"  # (max token = 128000)
    # model_coder = "gpt4olatest"  # (max token = 128000)
    # model = "gpt4olatest"  # (max token = 200000)
    # model_coder = "gpt4olatest"  # (max token = 200000)
    model = "gpt54"
    model_coder = "gpt54"
    model_judge = "claudesonnet46"  # (max token = 200000)

    # -- FROM CEREBRAS
    # model = "gpt-oss-120b"
    # model_judge = "gpt-oss-120b"
    # model_coder = "qwen-3-coder-480b"
    
    # -- FROM OLLAMA
    # model = "devstral:24b" # (max token = 128000)
    # model_coder = "devstral:24b" # (max token = 128000)
    # model_judge = "devstral:24b" # (max token = 128000)
    # model = "qwq:latest"  # (max token = 128000)
    # model_coder = "qwen2.5-coder:32b"  # (max token = 32768)
    # model_judge = "qwq:latest"  # (max token = 128000)
    # model = "qwq:latest"  # (max token = 131072)
    # model = "gemma3:27b"  # crashing on code gen
    # model_coder = "gemma3:27b"
    # model = "granite3.2:8b-instruct-q8_0"
    # model_coder = "granite-code:34b-instruct-q8_0"
    # model = "deepseek-r1:70b"  # (max token = 128000)
    # model_coder = "deepseek-r1:70b"  # (max token = 128000)
    # model_coder = "deepseek-coder-v2:16b"  # (max token = 131072)
    # model_coder = "deepseek-coder-v2:16b"  # (max token = 131072)
    # model = "dolphin3"  # (max token = 131072)
    # model = "llama3.3:latest"  # (max token = 131072)
    # model = "phi4" # (max token = 16384)
    # model_coder = "codestral:22b"  # (max token = 32768)
    # model = "codestral:22b-v0.1-q8_0"  # (max token = 32768)
    # model = "wizardcoder:33b-v1.1-q8_0"  # (max token = 16384)
    # model_coder = "deepseek-coder:33b-instruct-q8_0"  # (max token = 16384)
    # model = "deepseek-coder-v2:16b-lite-instruct-fp16"  # (max token = 128000)
    # model = "deepseek-coder-v2:16b-lite-instruct-q8_0"
    # model = "phi3:14b"  # (max token = 131072)
    # model_judge = "llama3.3:70b"  # (max token = 128000)
    # model = "qwen3:latest"
    # model_coder = "qwen3:latest"
    # model_judge = "qwen3:latest"  # (max token = 40960)
    # model_coder = "qwen3-coder:30b"  # context 262144
    # model = "gpt-oss:20b"
    # model_judge = "gpt-oss:20b"
  
    # Set maximum context token count size for selected LLM:
    num_ctx = 200000
    num_ctx_coder = 200000

    # Needed only for Argo endpoints
    uri = "local"  # local  remote

    # ASSIGN LLM API SOURCE:
    if model in ["gpt54", "gpt5", "gpt35", "gpt35large", "gpt4", "gpt4large", "gpt4turbo", "gpt4o", "gpt4olatest", "gpt41",
                 "gpto1preview", "gpto1mini", "gpto3mini", "gpto4mini", "claudesonnet4", "claudesonnet45"]:
        llm_source = "argo"
    elif model in ["gpt-oss-120b", "qwen-3-coder-480b"]:
        llm_source = "cerebras"
    elif ("GGUF" in model and "hf.co" not in model):
        llm_source = "lmstudio"
    else:
        llm_source = "ollama"

    """
    [4] SET ADDITIONAL PIPELINE PARAMETERS
    """
    experiment = "TEST_FULLEXP_verify_0314-2026_" + model_coder + "_" + app_folder_name + "_runtm_" + detected_vendor.upper()  # Assign any label for experiment name.
    display_response = True
    test_with_error = False
    generate_test_case = False

    # Unload any previously loaded LLMs in the Ollama server 
    # (clear out the memory)
    if llm_source == "ollama":
        lassi.llm_config.unload_ollama_model(model)
        lassi.llm_config.unload_ollama_model(model_coder)

    """
    [5] SET DATA COLLECTION CONFIGURATION
    """
    # Data spreadsheet column headers
    DATAFIELDNAMES = [
        "trial_number",
        "timestamp",
        "experiment",
        "gpu_vendor",
        "app_name",
        "code_line_count",
        "llm",
        "llm_coder",
        "llm_judge",
        "source_v0_energy_J",
        "source_v0_avg_power_W",
        "source_v0_runtime_s",
        "source_verify_runtime_s",  # ★ instrumented source runtime
        "base_v1_energy_J",
        "base_v1_avg_power_W",
        "base_v1_runtime_s",
        "base_v1_valid",
        "base_v1_energy_diff",
        "base_v1_avg_power_diff",
        "base_v1_runtime_diff",
        "zeroshot_runtime_s",        # ★ schema-aligned alias for base_v1_runtime_s
        "zeroshot_compile_status",   # ★ "success" | "compile_error" | "skipped"
        "zeroshot_execute_status",   # ★ "success" | "execute_error" | "skipped"
        # "base_v1_optimization_methods",
        # "base_v2_energy_J",
        # "base_v2_avg_power_W",
        # "base_v2_runtime_s",
        # "base_v2_valid",
        # "base_v2_selfcorrections",
        # "base_v2_energy_diff",
        # "base_v2_avg_power_diff",
        # "base_v2_runtime_diff",
        # "base_v2_optimization_methods",
        "lassiee_energy_J",
        "lassiee_avg_power_W",
        "lassiee_runtime_s",
        "lassiee_valid",
        "lassiee_version",
        "lassiee_llm_temp",
        "lassiee_selfcorrections",
        "lassiee_eeiterations",
        "lassiee_energy_diff",
        "lassiee_avg_power_diff",
        "lassiee_runtime_diff",
        "lassiee_optimization_methods",
        "total_selfcorrections",
        "compile_selfcorrections",   # ★
        "execute_selfcorrections",   # ★
        "tier1_selfcorrections",     # ★
        "tier2_selfcorrections",     # ★
        "tier3_selfcorrections",     # ★
        "global_counter_at_end",     # ★ global verify counter at trial end
        "total_code_versions",       # ★ distinct LASSI code block indices
        "total_code_attempts",       # ★ rows in attempt log for this trial
        "total_eeiterations",
        "pipeline_runtime_min",
        "tier2_pass_count",
        "tier2_fail_total",
        "tier3_pass_count",
        "tier3_fail_total",
        "tier3_active_relations",
        "tier3_inactive_relations",  # ★
        "judge_false_positives",
        "judge_false_negatives",
        "lassiee_clean_status",
        "lassiee_clean_runtime_s",
        "lassiee_clean_runtime_diff",
        "lassiee_clean_energy_J",
        "lassiee_clean_energy_diff",
        "base_llm_tier1_verdict",
        "base_llm_tier2_verdict",
        "base_llm_tier3_verdict",
        "base_llm_verified_valid"
    ]
    
    return (app_folder_name, app_name, file_name, lines_of_code, source_lang, target_lang, model, model_coder, model_judge, num_ctx, num_ctx_coder, llm_source, use_context, use_context_summaries_only,
            few_shot_learning, run_code_analysis, context_knowledge_selection, source_folder, code_compiler, code_compiler_kwds, build_script, make_script, code_build_script, yes_execute, exe_input_parameters,
            experiment, display_response, test_with_error, generate_test_case, power_measure, yes_power_feedback, baseline_oneshot_goal, use_self_prompting, vary_llm_temp,
            verifyprompt, DATAFIELDNAMES, detected_vendor.upper(),
            run_tier2, tier2_tolerance, run_tier3, tier3_tolerance,
            run_llm_judge)

# EXPERIMENTAL DATA COLLETION MANAGEMENT
def init_data_collection(dir_path, DATAFIELDNAMES, filename="result_lassi-verify_pipeline_data.csv"):
    os.makedirs(dir_path, exist_ok=True)
    fp = os.path.join(dir_path, filename)
    if not os.path.exists(fp):
        with open(fp, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=DATAFIELDNAMES)
            writer.writeheader()
    return fp


_____
***STEP 2/2: Run the following block to implement the pipeline with the configuration parameters.***

In [5]:
# Experimental repeat for pass@k metrics
trial_number = 1
total_trials = 1

# Load prompt dictionary (to capture recent manual changes to script, if any)
prompts = PromptDictionary()

all_trials_start = time.time()

# Store the current home working directory (needed for code execution)
original_home_directory = os.getcwd()


# COLLECT THE EXPERIMENT CONFIGURATIONS
(
    app_folder_name, app_name, file_name, lines_of_code, source_lang, target_lang, model, model_coder, model_judge, num_ctx, num_ctx_coder,
    llm_source, use_context, use_context_summaries_only, few_shot_learning, run_code_analysis, context_knowledge_selection, source_folder, code_compiler,
    code_compiler_kwds, build_script, make_script, code_build_script, yes_execute, exe_input_parameters, experiment,
    display_response, test_with_error, generate_test_case, power_measure, yes_power_feedback, baseline_oneshot_goal,
    use_self_prompting, vary_llm_temp, verifyprompt, DATAFIELDNAMES, gpu_vendor,
    run_tier2, tier2_tolerance, run_tier3, tier3_tolerance,
    run_llm_judge
) = experimental_setup()

# IMPORT CONTEXT KNOWLEDGE FOR TARGET PROGRAMMING LANGUAGE
if gpu_vendor == "NVIDIA":
    input_context = "[BACKGROUND KNOWLEDGE: CUDA C++ PROGRAMMING BEST PRACTICES] " + prompts.get_codeknowledge("cuda") + "\n\n"  # [/END GENERAL CUDA PROGRAMMING CONTEXT]
    input_context = input_context + "[BACKGROUND KNOWLEDGE: CUDA C++ OPTIMIZATION GUIDE] " + prompts.get_codeknowledge(str(context_knowledge_selection)) + "\n\n"  #  [/END CUDA OPTIMIZATION CONTEXT]
elif gpu_vendor == "AMD":
    input_context = "[BACKGROUND KNOWLEDGE: HIP C++ PROGRAMMING BEST PRACTICES] " + prompts.get_codeknowledge("hip") + "\n\n"  # [/END GENERAL CUDA PROGRAMMING CONTEXT]
    input_context = input_context + "[BACKGROUND KNOWLEDGE: HIP C++ OPTIMIZATION GUIDE] " + prompts.get_codeknowledge(str(context_knowledge_selection)) + "\n\n"  #  [/END CUDA OPTIMIZATION CONTEXT]


# DEFINE PROMPT TEMPLATES
# TODO: MOST TASK PROMPT TEMPLATES WILL BE HANDLED BY THE PIPELINE METHODS
# If we are doing the runtime task, then set the system prompts, otherwise they are for ee
# we need to bring back in the translation systems prompt reference when we create LASSI-X
if (power_measure == False and yes_power_feedback == True):
    system_prompt_selection_index = "general_system_" + source_lang + "_runtime"
else:
    system_prompt_selection_index = source_lang + "_ee"  # "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
this_llm_system_prompt = prompts.get_system_prompt(str(system_prompt_selection_index))

coderefactor_prompt_selection_index = source_lang  # + "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
if (not baseline_oneshot_goal and use_self_prompting):
    # TODO: THIS SHOULD BE A DEAD BRANCH BECAUSE WE DON'T WANT TO USE baseline_oneshot_goal ANYMORE
    his_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index))
else: 
    if (power_measure == False and yes_power_feedback == True):
        # RUNTIME TASK
        if use_context:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_runtime"))
        else:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_nocontext_runtime"))
    else:
        # EE TASK
        if use_context:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline"))
        else:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_nocontext"))

# LOAD INPUT CODE TO REFACTOR
input_code_folder = source_folder + "/" + app_folder_name + "/"
if build_script:
    input_code_filename = app_name
elif make_script:
    input_code_filename = file_name
else:
    input_code_filename = app_folder_name + "-" + source_lang.lower() + "_main"
if source_lang == "OMP" or source_lang == "SYCL":
    input_code_extension = "cpp"
elif source_lang == "CUDA" or source_lang == "HIP":
    if (make_script and source_lang == "HIP"):
        input_code_extension = "cpp"  # custom for miniMDock and XSBench
    else:
        input_code_extension = "cu"
if make_script:
    source_file_directory = "generated_codes/" + input_code_folder
else:
    source_file_directory = "source_codes/" + input_code_folder
if make_script:
    # SELECT FILE TO REFACTOR from miniapp
    if source_lang == "CUDA":
        if app_name == "miniMDock":
            input_code_file_subdirectory = "device/cuda/"
        elif app_name == "XSBench":
            input_code_file_subdirectory = ""
    elif source_lang == "HIP":
        if app_name == "miniMDock":
            input_code_file_subdirectory = "device/hip/"
        elif app_name == "XSBench":
            input_code_file_subdirectory = ""
    source_file_path = source_file_directory + input_code_file_subdirectory + input_code_filename + "." + input_code_extension
    source_file_path = os.path.abspath(source_file_path)
else:
    input_code_file_subdirectory = ""
    source_file_path = source_file_directory + input_code_filename + "." + input_code_extension
    source_file_path = os.path.abspath(source_file_path)
with open(source_file_path, 'r') as file:
    input_code = file.read()
file.close()


# SPECIFY OUTPUT CODE SAVE
if build_script:
    output_code_directory = "generated_codes/" + source_folder + "/" + app_folder_name
    output_code_file_name = app_name
elif make_script:
    output_code_file_name = file_name
    output_code_directory = "generated_codes/" + source_folder + "/" + app_folder_name
else:
    output_code_directory = "generated_codes/" + llm_source + "_" + model.replace('/', '--') + "/promptset_" + str(coderefactor_prompt_selection_index)
    output_code_file_name = input_code_filename + "_ee_refactorcode_" + source_lang + "_exp" + experiment

now = datetime.now(ZoneInfo("America/Chicago"))
dt_string = now.strftime("%m-%d-%Y_t%H_%M_%S")

# SPECIFIY DATA COLLECTION SAVE
experiment_data_directory = "generated_codes/experiment/" + experiment + "__" + dt_string
# Convert to absolute directory::
experiment_data_directory = os.path.abspath(experiment_data_directory)
experiment_data_file_object = init_data_collection(experiment_data_directory, DATAFIELDNAMES)
experiment_data = dict.fromkeys(DATAFIELDNAMES, None) # all keys start out as None
# Table 2 ★ per-attempt log — one row per code generation attempt (including self-correction retries)
attempt_log_file_path = init_data_collection(
    experiment_data_directory, lassi.pipeline_v4.ATTEMPT_LOG_FIELDNAMES,
    filename="result_lassi-verify_attempt_log.csv"
)
experiment_data["gpu_vendor"] = str(detected_vendor.upper())

while trial_number <= total_trials:
    # For the miniApp, we need to reset the source kernel file
    #  because the pipeline for this scenario must overwrite the 
    #  source code in the generated_codes folder.
    if make_script:
        if source_lang == "CUDA":
            if app_name == "miniMDock":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/cuda/kernels.cu"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/cuda/kernels_orig_merged.cu"
            elif app_name == "XSBench":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/cuda/Main.cu"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/cuda/Main_orig_merged.cu"
        elif source_lang == "HIP":
            if app_name == "miniMDock":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/hip/kernels.cpp"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/hip/kernels_orig_merged.cpp"
            elif app_name == "XSBench":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/hip/Main.cpp"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/hip/Main_orig_merged.cpp"
        try:
            print(f"[reset] Copying clean kernel -> working file\n"
                  f"        src: {clean_kernel_file}\n"
                  f"        dst: {working_kernel_file}")
            shutil.copyfile(clean_kernel_file, working_kernel_file)
            print("[reset] Success: working file reset to clean copy.")
        except FileNotFoundError as e:
            print(f"[reset] ERROR: file not found: {e.filename}")
        except PermissionError as e:
            print(f"[reset] ERROR: permission denied: {e.filename}")
        except OSError as e:
            # catches other IO errors (disk full, cross-device issues, etc.)
            print(f"[reset] ERROR: {e.__class__.__name__}: {e}")

    # CREATE METADATA RECORD CONTENT
    meta_data_file = "***** EXPERIMENT CONFIGURATION SUMMARY *****\n"
    meta_data_file += "\nGPU Vendor: " + str(detected_vendor.upper())
    meta_data_file += "\nDate-time: " + dt_string
    meta_data_file += "\nApplication: " + app_folder_name
    meta_data_file += "\n\nRefactoring task: " + system_prompt_selection_index
    meta_data_file += "\n\nLLM Source: " + llm_source
    meta_data_file += "\nModel: " + model
    meta_data_file += "\nModel coder: " + model_coder
    meta_data_file += "\nModel judge: " + model_judge
    meta_data_file += "\nSystem Prompt Index: " + str(coderefactor_prompt_selection_index)
    meta_data_file += "\nCode file template: " + output_code_directory + "/" + output_code_file_name
    meta_data_file += "\n\nLLM System Prompt: " + "\n\n" + this_llm_system_prompt
    # meta_data_file += "\n\nLLM Prompt for Code Generation: " + "\n\n" + this_llm_refactoredcode_prompt
    meta_data_file += "\n\nPower measurement: " + str(power_measure)
    
    # Display metadata
    print("\n" + meta_data_file + "\n")
    
    print("\n\n" + color.BLUE + color.BOLD + "***** BEGIN THE LASSI ENERGY EFFICIENCY PIPELINE (v4) *****" + color.END + " Trial #: " + str(trial_number) + "\n\n")
    
    # RUN LASSI-EE PIPELINE
    complete = lassi.pipeline_v4.auto_code_llmgeneration_pipeline(
        prompts,
        this_llm_system_prompt,
        this_llm_refactoredcode_prompt,
        llm_source, 
        model,
        model_coder,
        model_judge,
        num_ctx,
        num_ctx_coder,
        # arrayormatrix_size_range,
        # thread_count_range,
        use_context,
        use_context_summaries_only,
        few_shot_learning,
        run_code_analysis,
        input_context,
        input_code_file_subdirectory,
        input_code,
        lines_of_code,
        # target_code,
        target_lang,
        input_code_extension,
        source_lang,
        code_compiler,
        code_compiler_kwds,
        build_script,
        make_script,
        code_build_script,
        output_code_directory,
        output_code_file_name,
        source_file_path,
        display_response,
        test_with_error,
        generate_test_case,
        yes_execute,
        exe_input_parameters,
        meta_data_file,
        power_measure,
        yes_power_feedback,
        dt_string,
        baseline_oneshot_goal,
        use_self_prompting,
        vary_llm_temp,
        experiment,
        verifyprompt,
        trial_number,
        experiment_data_directory,
        experiment_data_file_object,
        experiment_data,
        app_name,
        run_tier2=run_tier2,
        tier2_tolerance=tier2_tolerance,
        run_tier3=run_tier3,
        tier3_tolerance=tier3_tolerance,
        run_llm_judge=run_llm_judge,
        attempt_log_file_path=attempt_log_file_path
    )
    
    # Reset to original working directory
    os.chdir(original_home_directory)
    
    if complete:
        print("\n\n" + color.GREEN + color.BOLD + "***** LASSI PIPELINE COMPLETED *****\n\n" + color.END)
    else:
        print("\n\n" + color.RED + color.BOLD + "***** LASSI PIPELINE FAILED TO COMPLETE *****\n\n" + color.END)

    # Increment experiment counter
    trial_number = trial_number + 1


all_trials_end = time.time() - all_trials_start

print("\n\n" + color.PURPLE + color.BOLD + "***** ALL TRIALS COMPLETED ***** Total time (min): " + str(round(all_trials_end/60, 2)) + "\n\n" + color.END)




***** EXPERIMENT CONFIGURATION SUMMARY *****

GPU Vendor: NVIDIA
Date-time: 03-15-2026_t17_03_26
Application: lid-driven-cavity

Refactoring task: general_system_CUDA_runtime

LLM Source: argo
Model: gpt54
Model coder: gpt54
Model judge: claudesonnet46
System Prompt Index: CUDA
Code file template: generated_codes/argo_gpt54/promptset_CUDA/lid-driven-cavity-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gpt54_lid-driven-cavity_runtm_NVIDIA

LLM System Prompt: 

You are an expert C++/CUDA engineer. Your goal is to refactor provided CUDA C++ code to reduce wall-clock execution time by improving GPU and CPU utilization and parallelism, while preserving all original functionality and expected output of the provided code. You must also carefully read and incorporate the best practices and CUDA optimization techniques from the provided background documents and refactoring plans. You always surround your generated code with the three characters ```.

Power measurement: False


Waiting for the system to settle down for 1 seconds...: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it]


executable_file:  /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/lid-driven-cavity/lid-driven-cavity-cuda_main
initial_dir:  /home/mdearing/Research/LASSI/LASSI-Verify
directory_path:  /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/lid-driven-cavity
executable:  lid-driven-cavity-cuda_main
Working directory: /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/lid-driven-cavity
Execute command: ['./lid-driven-cavity-cuda_main', '']

running: executable -  lid-driven-cavity-cuda_main

----- < BEGIN: STANDARD OUTPUT FROM CODE EXECUTION > -----

Problem size: 512 x 512 
Time = 0.000477, delt = 4.768372e-04, iter = 1, res = 0.000000e+00
Time = 0.000954, delt = 4.768372e-04, iter = 1, res = 0.000000e+00
Time = 0.001000, delt = 4.632568e-05, iter = 1, res = 0.000000e+00

Total execution time of the iteration loop: 0.000018 (s)

THE RETURN CODE:  0

----- < END: Standard Output > -----


Program executed successfully!

----- END OF STAGE 0: TEST

```cpp
/** GPU solver for 2D lid-driven cavity problem, using finite difference method
 * \file main_gpu.cu
 *
 * \author Kyle E. Niemeyer
 * \date 09/27/2012
 *
 * Solve the incompressible, isothermal 2D Navier–Stokes equations for a square
 * lid-driven cavity on a GPU (via CUDA), using the finite difference method.
 * To change the grid resolution, modify "NUM". In addition, the problem is controlled
 * by the Reynolds number ("Re_num").
 * 
 * Based on the methodology given in Chapter 3 of "Numerical Simulation in Fluid
 * Dynamics", by M. Griebel, T. Dornseifer, and T. Neunhoeffer. SIAM, Philadelphia,
 * PA, 1998.
 * 
 * Boundary conditions:
 * u = 0 and v = 0 at x = 0, x = L, y = 0
 * u = ustar at y = H
 * v = 0 at y = H
 */

#include <stdlib.h>
#include <stdio.h>
#include <math.h>
#include <chrono>
#include <cuda.h>
#include <cstdio>
#include <cmath>

/** Problem size along one side; total number of cells is this squared */
#define NUM 512

// block size
#define BLOCK_SIZE 128

/** Double precision */
#define DOUBLE

#ifdef DOUBLE
#define Real double

#define ZERO 0.0
#define ONE 1.0
#define TWO 2.0
#define FOUR 4.0

#define SMALL 1.0e-10;

/** Reynolds number */
const Real Re_num = 1000.0;

/** SOR relaxation parameter */
const Real omega = 1.7;

/** Discretization mixture parameter (gamma) */
const Real mix_param = 0.9;

/** Safety factor for time step modification */
const Real tau = 0.5;

/** Body forces in x- and y- directions */
const Real gx = 0.0;
const Real gy = 0.0;

/** Domain size (non-dimensional) */
#define xLength 1.0
#define yLength 1.0
#else
#define Real float

// replace double functions with float versions
#undef fmin
#define fmin fminf
#undef fmax
#define fmax fmaxf
#undef fabs
#define fabs fabsf
#undef sqrt
#define sqrt sqrtf

#define ZERO 0.0f
#define ONE 1.0f
#define TWO 2.0f
#define FOUR 4.0f
#define SMALL 1.0e-10f;

/** Reynolds number */
const Real Re_num = 1000.0f;

/** SOR relaxation parameter */
const Real omega = 1.7f;

/** Discretization mixture parameter (gamma) */
const Real mix_param = 0.9f;

/** Safety factor for time step modification */
const Real tau = 0.5f;

/** Body forces in x- and y- directions */
const Real gx = 0.0f;
const Real gy = 0.0f;

/** Domain size (non-dimensional) */
#define xLength 1.0f
#define yLength 1.0f
#endif

/** Mesh sizes */
const Real dx = xLength / NUM;
const Real dy = yLength / NUM;

/** Max macro (type safe, from GNU) */
//#define MAX(a,b) ({ __typeof__ (a) _a = (a); __typeof__ (b) _b = (b); _a > _b ? _a : _b; })

/** Min macro (type safe) */
//#define MIN(a,b) ({ __typeof__ (a) _a = (a); __typeof__ (b) _b = (b); _a < _b ? _a : _b; })

// map two-dimensional indices to one-dimensional memory
#define u(I, J) u[((I) * ((NUM) + 2)) + (J)]
#define v(I, J) v[((I) * ((NUM) + 2)) + (J)]
#define F(I, J) F[((I) * ((NUM) + 2)) + (J)]
#define G(I, J) G[((I) * ((NUM) + 2)) + (J)]
#define pres_red(I, J) pres_red[((I) * ((NUM_2) + 2)) + (J)]
#define pres_black(I, J) pres_black[((I) * ((NUM_2) + 2)) + (J)]

///////////////////////////////////////////////////////////////////////////////
__host__
void set_BCs_host (Real* u, Real* v) 
{
  int ind;

  // loop through rows and columns
  for (ind = 0; ind < NUM + 2; ++ind) {

    // left boundary
    u(0, ind) = ZERO;
    v(0, ind) = -v(1, ind);

    // right boundary
    u(NUM, ind) = ZERO;
    v(NUM + 1, ind) = -v(NUM, ind);

    // bottom boundary
    u(ind, 0) = -u(ind, 1);
    v(ind, 0) = ZERO;

    // top boundary
    u(ind, NUM + 1) = TWO - u(ind, NUM);
    v(ind, NUM) = ZERO;

    if (ind == NUM) {
      // left boundary
      u(0, 0) = ZERO;
      v(0, 0) = -v(1, 0);
      u(0, NUM + 1) = ZERO;
      v(0, NUM + 1) = -v(1, NUM + 1);

      // right boundary
      u(NUM, 0) = ZERO;
      v(NUM + 1, 0) = -v(NUM, 0);
      u(NUM, NUM + 1) = ZERO;
      v(NUM + 1, NUM + 1) = -v(NUM, NUM + 1);

      // bottom boundary
      u(0, 0) = -u(0, 1);
      v(0, 0) = ZERO;
      u(NUM + 1, 0) = -u(NUM + 1, 1);
      v(NUM + 1, 0) = ZERO;

      // top boundary
      u(0, NUM + 1) = TWO - u(0, NUM);
      v(0, NUM) = ZERO;
      u(NUM + 1, NUM + 1) = TWO - u(NUM + 1, NUM);
      v(ind, NUM + 1) = ZERO;
    } // end if

  } // end for

} // end set_BCs_host

///////////////////////////////////////////////////////////////////////////////
__global__
void set_BCs (Real*__restrict__ u, Real*__restrict__ v) 
{
  int ind = (blockIdx.x * blockDim.x) + threadIdx.x + 1;

  // left boundary
  u(0, ind) = ZERO;
  v(0, ind) = -v(1, ind);

  // right boundary
  u(NUM, ind) = ZERO;
  v(NUM + 1, ind) = -v(NUM, ind);

  // bottom boundary
  u(ind, 0) = -u(ind, 1);
  v(ind, 0) = ZERO;

  // top boundary
  u(ind, NUM + 1) = TWO - u(ind, NUM);
  v(ind, NUM) = ZERO;

  if (ind == NUM) {
    // left boundary
    u(0, 0) = ZERO;
    v(0, 0) = -v(1, 0);
    u(0, NUM + 1) = ZERO;
    v(0, NUM + 1) = -v(1, NUM + 1);

    // right boundary
    u(NUM, 0) = ZERO;
    v(NUM + 1, 0) = -v(NUM, 0);
    u(NUM, NUM + 1) = ZERO;
    v(NUM + 1, NUM + 1) = -v(NUM, NUM + 1);

    // bottom boundary
    u(0, 0) = -u(0, 1);
    v(0, 0) = ZERO;
    u(NUM + 1, 0) = -u(NUM + 1, 1);
    v(NUM + 1, 0) = ZERO;

    // top boundary
    u(0, NUM + 1) = TWO - u(0, NUM);
    v(0, NUM) = ZERO;
    u(NUM + 1, NUM + 1) = TWO - u(NUM + 1, NUM);
    v(ind, NUM + 1) = ZERO;
  } // end if

} // end set_BCs

///////////////////////////////////////////////////////////////////////////////

__global__ 
void calculate_F (const Real dt,
                  const Real*__restrict__ u,
                  const Real*__restrict__ v,
                        Real*__restrict__ F) 
{  
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  if (col == NUM) {
    // right boundary, F_ij = u_ij
    // also do left boundary
    F(0, row) = u(0, row);
    F(NUM, row) = u(NUM, row);
  } else {

    // u velocities
    Real u_ij = u(col, row);
    Real u_ip1j = u(col + 1, row);
    Real u_ijp1 = u(col, row + 1);
    Real u_im1j = u(col - 1, row);
    Real u_ijm1 = u(col, row - 1);

    // v velocities
    Real v_ij = v(col, row);
    Real v_ip1j = v(col + 1, row);
    Real v_ijm1 = v(col, row - 1);
    Real v_ip1jm1 = v(col + 1, row - 1);

    // finite differences
    Real du2dx, duvdy, d2udx2, d2udy2;

    du2dx = (((u_ij + u_ip1j) * (u_ij + u_ip1j) - (u_im1j + u_ij) * (u_im1j + u_ij))
        + mix_param * (fabs(u_ij + u_ip1j) * (u_ij - u_ip1j)
          - fabs(u_im1j + u_ij) * (u_im1j - u_ij))) / (FOUR * dx);
    duvdy = ((v_ij + v_ip1j) * (u_ij + u_ijp1) - (v_ijm1 + v_ip1jm1) * (u_ijm1 + u_ij)
        + mix_param * (fabs(v_ij + v_ip1j) * (u_ij - u_ijp1)
          - fabs(v_ijm1 + v_ip1jm1) * (u_ijm1 - u_ij))) / (FOUR * dy);
    d2udx2 = (u_ip1j - (TWO * u_ij) + u_im1j) / (dx * dx);
    d2udy2 = (u_ijp1 - (TWO * u_ij) + u_ijm1) / (dy * dy);

    F(col, row) = u_ij + dt * (((d2udx2 + d2udy2) / Re_num) - du2dx - duvdy + gx);

  } // end if

} // end calculate_F

///////////////////////////////////////////////////////////////////////////////

__global__ 
void calculate_G (const Real dt,
                  const Real*__restrict__ u,
                  const Real*__restrict__ v,
                        Real*__restrict__ G) 
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  if (row == NUM) {
    // top and bottom boundaries
    G(col, 0) = v(col, 0);
    G(col, NUM) = v(col, NUM);

  } else {

    // u velocities
    Real u_ij = u(col, row);
    Real u_ijp1 = u(col, row + 1);
    Real u_im1j = u(col - 1, row);
    Real u_im1jp1 = u(col - 1, row + 1);

    // v velocities
    Real v_ij = v(col, row);
    Real v_ijp1 = v(col, row + 1);
    Real v_ip1j = v(col + 1, row);
    Real v_ijm1 = v(col, row - 1);
    Real v_im1j = v(col - 1, row);

    // finite differences
    Real dv2dy, duvdx, d2vdx2, d2vdy2;

    dv2dy = ((v_ij + v_ijp1) * (v_ij + v_ijp1) - (v_ijm1 + v_ij) * (v_ijm1 + v_ij)
        + mix_param * (fabs(v_ij + v_ijp1) * (v_ij - v_ijp1)
          - fabs(v_ijm1 + v_ij) * (v_ijm1 - v_ij))) / (FOUR * dy);
    duvdx = ((u_ij + u_ijp1) * (v_ij + v_ip1j) - (u_im1j + u_im1jp1) * (v_im1j + v_ij)
        + mix_param * (fabs(u_ij + u_ijp1) * (v_ij - v_ip1j) 
          - fabs(u_im1j + u_im1jp1) * (v_im1j - v_ij))) / (FOUR * dx);
    d2vdx2 = (v_ip1j - (TWO * v_ij) + v_im1j) / (dx * dx);
    d2vdy2 = (v_ijp1 - (TWO * v_ij) + v_ijm1) / (dy * dy);

    G(col, row) = v_ij + dt * (((d2vdx2 + d2vdy2) / Re_num) - dv2dy - duvdx + gy);

  } // end if

} // end calculate_G

///////////////////////////////////////////////////////////////////////////////

__global__ 
void sum_pressure (const Real*__restrict__ pres_red,
                   const Real*__restrict__ pres_black, 
                         Real*__restrict__ pres_sum) 
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  // shared memory for block's sum
  __shared__ Real sum_cache[BLOCK_SIZE];

  int NUM_2 = NUM >> 1;

  Real pres_r = pres_red(col, row);
  Real pres_b = pres_black(col, row);

  // add squared pressure
  sum_cache[threadIdx.x] = (pres_r * pres_r) + (pres_b * pres_b);

  // synchronize threads in block to ensure all thread values stored
  __syncthreads();

  // add up values for block
  int i = BLOCK_SIZE >> 1;
  while (i != 0) {
    if (threadIdx.x < i) {
      sum_cache[threadIdx.x] += sum_cache[threadIdx.x + i];
    }
    __syncthreads();
    i >>= 1;
  }

  // store block's summed values
  if (threadIdx.x == 0) {
    pres_sum[blockIdx.y + (gridDim.y * blockIdx.x)] = sum_cache[0];
  }

} // end sum_pressure

///////////////////////////////////////////////////////////////////////////////
__global__ 
void set_horz_pres_BCs (Real*__restrict__ pres_red, Real*__restrict__ pres_black) 
{
  int col = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  col = (col * 2) - 1;

  int NUM_2 = NUM >> 1;

  // p_i,0 = p_i,1
  pres_black(col, 0) = pres_red(col, 1);
  pres_red(col + 1, 0) = pres_black(col + 1, 1);

  // p_i,jmax+1 = p_i,jmax
  pres_red(col, NUM_2 + 1) = pres_black(col, NUM_2);
  pres_black(col + 1, NUM_2 + 1) = pres_red(col + 1, NUM_2);

} // end set_horz_pres_BCs

//////////////////////////////////////////////////////////////////////////////

__global__
void set_vert_pres_BCs (Real*__restrict__ pres_red, Real*__restrict__ pres_black) 
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;

  int NUM_2 = NUM >> 1;

  // p_0,j = p_1,j
  pres_black(0, row) = pres_red(1, row);
  pres_red(0, row) = pres_black(1, row);

  // p_imax+1,j = p_imax,j
  pres_black(NUM + 1, row) = pres_red(NUM, row);
  pres_red(NUM + 1, row) = pres_black(NUM, row);

} // end set_pressure_BCs

///////////////////////////////////////////////////////////////////////////////

/** Function to update pressure for red cells
 * 
 * \param[in]    dt      time-step size
 * \param[in]    F      array of discretized x-momentum eqn terms
 * \param[in]    G      array of discretized y-momentum eqn terms
 * \param[in]    pres_black  pressure values of black cells
 * \param[inout]  pres_red  pressure values of red cells
 */
__global__
void red_kernel (const Real dt,
                 const Real*__restrict__ F, 
                 const Real*__restrict__ G,
                 const Real*__restrict__ pres_black,
                       Real*__restrict__ pres_red) 
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  int NUM_2 = NUM >> 1;      

  Real p_ij = pres_red(col, row);

  Real p_im1j = pres_black(col - 1, row);
  Real p_ip1j = pres_black(col + 1, row);
  Real p_ijm1 = pres_black(col, row - (col & 1));
  Real p_ijp1 = pres_black(col, row + ((col + 1) & 1));

  // right-hand side
  Real rhs = (((F(col, (2 * row) - (col & 1))
          - F(col - 1, (2 * row) - (col & 1))) / dx)
      + ((G(col, (2 * row) - (col & 1))
          - G(col, (2 * row) - (col & 1) - 1)) / dy)) / dt;

  pres_red(col, row) = p_ij * (ONE - omega) + omega * 
    (((p_ip1j + p_im1j) / (dx * dx)) + ((p_ijp1 + p_ijm1) / (dy * dy)) - 
     rhs) / ((TWO / (dx * dx)) + (TWO / (dy * dy)));

} // end red_kernel

///////////////////////////////////////////////////////////////////////////////

/** Function to update pressure for black cells
 * 
 * \param[in]    dt      time-step size
 * \param[in]    F      array of discretized x-momentum eqn terms
 * \param[in]    G      array of discretized y-momentum eqn terms
 * \param[in]    pres_red  pressure values of red cells
 * \param[inout]  pres_black  pressure values of black cells
 */
__global__ 
void black_kernel (const Real dt,
                   const Real*__restrict__ F, 
                   const Real*__restrict__ G,
                   const Real*__restrict__ pres_red, 
                         Real*__restrict__ pres_black) 
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  int NUM_2 = NUM >> 1;

  Real p_ij = pres_black(col, row);

  Real p_im1j = pres_red(col - 1, row);
  Real p_ip1j = pres_red(col + 1, row);
  Real p_ijm1 = pres_red(col, row - ((col + 1) & 1));
  Real p_ijp1 = pres_red(col, row + (col & 1));

  // right-hand side
  Real rhs = (((F(col, (2 * row) - ((col + 1) & 1))
          - F(col - 1, (2 * row) - ((col + 1) & 1))) / dx)
      + ((G(col, (2 * row) - ((col + 1) & 1))
          - G(col, (2 * row) - ((col + 1) & 1) - 1)) / dy)) / dt;

  pres_black(col, row) = p_ij * (ONE - omega) + omega * 
    (((p_ip1j + p_im1j) / (dx * dx)) + ((p_ijp1 + p_ijm1) / (dy * dy)) - 
     rhs) / ((TWO / (dx * dx)) + (TWO / (dy * dy)));

} // end black_kernel

///////////////////////////////////////////////////////////////////////////////

__global__
void calc_residual (const Real dt,
                    const Real*__restrict__ F,
                    const Real*__restrict__ G, 
                    const Real*__restrict__ pres_red,
                    const Real*__restrict__ pres_black,
                          Real*__restrict__ res_array)
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  int NUM_2 = NUM >> 1;

  Real p_ij, p_im1j, p_ip1j, p_ijm1, p_ijp1, rhs, res, res2;

  // red point
  p_ij = pres_red(col, row);

  p_im1j = pres_black(col - 1, row);
  p_ip1j = pres_black(col + 1, row);
  p_ijm1 = pres_black(col, row - (col & 1));
  p_ijp1 = pres_black(col, row + ((col + 1) & 1));

  rhs = (((F(col, (2 * row) - (col & 1)) - F(col - 1, (2 * row) - (col & 1))) / dx)
      +  ((G(col, (2 * row) - (col & 1)) - G(col, (2 * row) - (col & 1) - 1)) / dy)) / dt;

  // calculate residual
  res = ((p_ip1j - (TWO * p_ij) + p_im1j) / (dx * dx))
    + ((p_ijp1 - (TWO * p_ij) + p_ijm1) / (dy * dy)) - rhs;

  // black point
  p_ij = pres_black(col, row);

  p_im1j = pres_red(col - 1, row);
  p_ip1j = pres_red(col + 1, row);
  p_ijm1 = pres_red(col, row - ((col + 1) & 1));
  p_ijp1 = pres_red(col, row + (col & 1));

  // right-hand side
  rhs = (((F(col, (2 * row) - ((col + 1) & 1)) - F(col - 1, (2 * row) - ((col + 1) & 1))) / dx)
      +  ((G(col, (2 * row) - ((col + 1) & 1)) - G(col, (2 * row) - ((col + 1) & 1) - 1)) / dy)) / dt;

  // calculate residual
  res2 = ((p_ip1j - (TWO * p_ij) + p_im1j) / (dx * dx))
    + ((p_ijp1 - (TWO * p_ij) + p_ijm1) / (dy * dy)) - rhs;

  // shared memory for block's sum
  __shared__ Real sum_cache[BLOCK_SIZE];

  sum_cache[threadIdx.x] = (res * res) + (res2 * res2);

  // synchronize threads in block to ensure all residuals stored
  __syncthreads();

  // add up squared residuals for block
  int i = BLOCK_SIZE >> 1;
  while (i != 0) {
    if (threadIdx.x < i) {
      sum_cache[threadIdx.x] += sum_cache[threadIdx.x + i];
    }
    __syncthreads();
    i >>= 1;
  }

  // store block's summed residuals
  if (threadIdx.x == 0) {
    res_array[blockIdx.y + (gridDim.y * blockIdx.x)] = sum_cache[0];
  }
} 

///////////////////////////////////////////////////////////////////////////////

__global__ 
void calculate_u (const Real dt,
                  const Real*__restrict__ F, 
                  const Real*__restrict__ pres_red,
                  const Real*__restrict__ pres_black, 
                        Real*__restrict__ u,
                        Real*__restrict__ max_u)
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  // allocate shared memory to store max velocities
  __shared__ Real max_cache[BLOCK_SIZE];
  max_cache[threadIdx.x] = ZERO;

  int NUM_2 = NUM >> 1;
  Real new_u = ZERO;

  if (col != NUM) {

    Real p_ij, p_ip1j, new_u2;

    // red point
    p_ij = pres_red(col, row);
    p_ip1j = pres_black(col + 1, row);

    new_u = F(col, (2 * row) - (col & 1)) - (dt * (p_ip1j - p_ij) / dx);
    u(col, (2 * row) - (col & 1)) = new_u;

    // black point
    p_ij = pres_black(col, row);
    p_ip1j = pres_red(col + 1, row);

    new_u2 = F(col, (2 * row) - ((col + 1) & 1)) - (dt * (p_ip1j - p_ij) / dx);
    u(col, (2 * row) - ((col + 1) & 1)) = new_u2;

    // check for max of these two
    new_u = fmax(fabs(new_u), fabs(new_u2));

    if ((2 * row) == NUM) {
      // also test for max velocity at vertical boundary
      new_u = fmax(new_u, fabs( u(col, NUM + 1) ));
    }
  } else {
    // check for maximum velocity in boundary cells also
    new_u = fmax(fabs( u(NUM, (2 * row)) ), fabs( u(0, (2 * row)) ));
    new_u = fmax(fabs( u(NUM, (2 * row) - 1) ), new_u);
    new_u = fmax(fabs( u(0, (2 * row) - 1) ), new_u);

    new_u = fmax(fabs( u(NUM + 1, (2 * row)) ), new_u);
    new_u = fmax(fabs( u(NUM + 1, (2 * row) - 1) ), new_u);

  } // end if

  // store maximum u for block from each thread
  max_cache[threadIdx.x] = new_u;

  // synchronize threads in block to ensure all velocities stored
  __syncthreads();

  // calculate maximum for block
  int i = BLOCK_SIZE >> 1;
  while (i != 0) {
    if (threadIdx.x < i) {
      max_cache[threadIdx.x] = fmax(max_cache[threadIdx.x], max_cache[threadIdx.x + i]);
    }
    __syncthreads();
    i >>= 1;
  }

  // store block's maximum
  if (threadIdx.x == 0) {
    max_u[blockIdx.y + (gridDim.y * blockIdx.x)] = max_cache[0];
  }
} // end calculate_u

///////////////////////////////////////////////////////////////////////////////

__global__ 
void calculate_v (const Real dt,
                  const Real*__restrict__ G, 
                  const Real*__restrict__ pres_red,
                  const Real*__restrict__ pres_black, 
                        Real*__restrict__ v,
                        Real*__restrict__ max_v)
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  // allocate shared memory to store maximum velocities
  __shared__ Real max_cache[BLOCK_SIZE];
  max_cache[threadIdx.x] = ZERO;

  int NUM_2 = NUM >> 1;
  Real new_v = ZERO;

  if (row != NUM_2) {
    Real p_ij, p_ijp1, new_v2;

    // red pressure point
    p_ij = pres_red(col, row);
    p_ijp1 = pres_black(col, row + ((col + 1) & 1));

    new_v = G(col, (2 * row) - (col & 1)) - (dt * (p_ijp1 - p_ij) / dy);
    v(col, (2 * row) - (col & 1)) = new_v;

    // black pressure point
    p_ij = pres_black(col, row);
    p_ijp1 = pres_red(col, row + (col & 1));

    new_v2 = G(col, (2 * row) - ((col + 1) & 1)) - (dt * (p_ijp1 - p_ij) / dy);
    v(col, (2 * row) - ((col + 1) & 1)) = new_v2;

    // check for max of these two
    new_v = fmax(fabs(new_v), fabs(new_v2));

    if (col == NUM) {
      // also test for max velocity at vertical boundary
      new_v = fmax(new_v, fabs( v(NUM + 1, (2 * row)) ));
    }

  } else {

    if ((col & 1) == 1) {
      // black point is on boundary, only calculate red point below it
      Real p_ij = pres_red(col, row);
      Real p_ijp1 = pres_black(col, row + ((col + 1) & 1));

      new_v = G(col, (2 * row) - (col & 1)) - (dt * (p_ijp1 - p_ij) / dy);
      v(col, (2 * row) - (col & 1)) = new_v;
    } else {
      // red point is on boundary, only calculate black point below it
      Real p_ij = pres_black(col, row);
      Real p_ijp1 = pres_red(col, row + (col & 1));

      new_v = G(col, (2 * row) - ((col + 1) & 1)) - (dt * (p_ijp1 - p_ij) / dy);
      v(col, (2 * row) - ((col + 1) & 1)) = new_v;
    }

    // get maximum v velocity
    new_v = fabs(new_v);

    // check for maximum velocity in boundary cells also
    new_v = fmax(fabs( v(col, NUM) ), new_v);
    new_v = fmax(fabs( v(col, 0) ), new_v);

    new_v = fmax(fabs( v(col, NUM + 1) ), new_v);
  } // end if

  // store absolute value of velocity
  max_cache[threadIdx.x] = new_v;

  // synchronize threads in block to ensure all velocities stored
  __syncthreads();

  // calculate maximum for block
  int i = BLOCK_SIZE >> 1;
  while (i != 0) {
    if (threadIdx.x < i) {
      max_cache[threadIdx.x] = fmax(max_cache[threadIdx.x], max_cache[threadIdx.x + i]);
    }
    __syncthreads();
    i >>= 1;
  }

  // store block's summed residuals
  if (threadIdx.x == 0) {
    max_v[blockIdx.y + (gridDim.y * blockIdx.x)] = max_cache[0];
  }
} // end calculate_v

///////////////////////////////////////////////////////////////////////////////

int main (int argc, char *argv[])
{
  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Entry parameter and compile-time configuration validity
  {
    bool _t1_pass = (NUM > 0 &&
                     BLOCK_SIZE > 0 &&
                     (NUM % BLOCK_SIZE) == 0 &&
                     ((NUM / 2) % BLOCK_SIZE) == 0 &&
                     Re_num > ZERO &&
                     omega > ZERO &&
                     tau > ZERO &&
                     tau <= ONE);
    printf("LASSI-VERIFY ASSERTIONS [main_entry_config_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // iterations for Red-Black Gauss-Seidel with SOR
  int iter = 0;
  const int it_max = 1000000;

  // SOR iteration tolerance
  const Real tol = 0.001;

  // time range
  const Real time_start = 0.0;
  const Real time_end = 0.001; //20.0;

  // initial time step size
  Real dt = 0.02;

  int size = (NUM + 2) * (NUM + 2);
  int size_pres = ((NUM / 2) + 2) * (NUM + 2);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Core scalar dimensions and time configuration validity
  {
    bool _t1_pass = (size > 0 &&
                     size_pres > 0 &&
                     it_max > 0 &&
                     tol > ZERO &&
                     time_end >= time_start &&
                     dt > ZERO &&
                     isfinite((double)dx) &&
                     isfinite((double)dy) &&
                     dx > ZERO &&
                     dy > ZERO);
    printf("LASSI-VERIFY ASSERTIONS [main_scalar_dims_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // arrays for pressure and velocity
  Real* F;
  Real* u;
  Real* G;
  Real* v;

  F = (Real *) calloc (size, sizeof(Real));
  u = (Real *) calloc (size, sizeof(Real));
  G = (Real *) calloc (size, sizeof(Real));
  v = (Real *) calloc (size, sizeof(Real));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Host allocation validity for velocity and intermediate arrays
  {
    bool _t1_pass = (F != NULL && u != NULL && G != NULL && v != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_host_ptrs_flow_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  for (int i = 0; i < size; ++i) {
    F[i] = ZERO;
    u[i] = ZERO;
    G[i] = ZERO;
    v[i] = ZERO;
  }

  // arrays for pressure
  Real* pres_red;
  Real* pres_black;

  pres_red = (Real *) calloc (size_pres, sizeof(Real));
  pres_black = (Real *) calloc (size_pres, sizeof(Real));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Host allocation validity for pressure arrays
  {
    bool _t1_pass = (pres_red != NULL && pres_black != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_host_ptrs_pressure_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  for (int i = 0; i < size_pres; ++i) {
    pres_red[i] = ZERO;
    pres_black[i] = ZERO;
  }

  // print problem size
  printf("Problem size: %d x %d \n", NUM, NUM);

  // block and grid dimensions

  // boundary conditions kernel
  dim3 block_bcs (BLOCK_SIZE, 1);
  dim3 grid_bcs (NUM / BLOCK_SIZE, 1);

  // pressure kernel
  dim3 block_pr (BLOCK_SIZE, 1);
  dim3 grid_pr (NUM / (2 * BLOCK_SIZE), NUM);

  // block and grid dimensions for F
  dim3 block_F (BLOCK_SIZE, 1);
  dim3 grid_F (NUM / BLOCK_SIZE, NUM);

  // block and grid dimensions for G
  dim3 block_G (BLOCK_SIZE, 1);
  dim3 grid_G (NUM / BLOCK_SIZE, NUM);

  // horizontal pressure boundary conditions
  dim3 block_hpbc (BLOCK_SIZE, 1);
  dim3 grid_hpbc (NUM / (2 * BLOCK_SIZE), 1);

  // vertical pressure boundary conditions
  dim3 block_vpbc (BLOCK_SIZE, 1);
  dim3 grid_vpbc (NUM / (2 * BLOCK_SIZE), 1);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Kernel launch configuration validity
  {
    bool _t1_pass = (block_bcs.x > 0 && block_bcs.x <= 1024 &&
                     block_pr.x > 0 && block_pr.x <= 1024 &&
                     block_F.x > 0 && block_F.x <= 1024 &&
                     block_G.x > 0 && block_G.x <= 1024 &&
                     block_hpbc.x > 0 && block_hpbc.x <= 1024 &&
                     block_vpbc.x > 0 && block_vpbc.x <= 1024 &&
                     grid_bcs.x > 0 &&
                     grid_pr.x > 0 && grid_pr.y > 0 &&
                     grid_F.x > 0 && grid_F.y > 0 &&
                     grid_G.x > 0 && grid_G.y > 0 &&
                     grid_hpbc.x > 0 &&
                     grid_vpbc.x > 0);
    printf("LASSI-VERIFY ASSERTIONS [main_kernel_params_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // residual variable
  Real* res;

  int size_res = grid_pr.x * grid_pr.y;
  res = (Real *) calloc (size_res, sizeof(Real));

  // variables to store maximum velocities
  Real* max_u_arr;
  Real* max_v_arr;
  int size_max = grid_pr.x * grid_pr.y;

  max_u_arr = (Real *) calloc (size_max, sizeof(Real));
  max_v_arr = (Real *) calloc (size_max, sizeof(Real));

  // pressure sum
  Real* pres_sum;
  pres_sum = (Real *) calloc (size_res, sizeof(Real));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Host allocation validity for reduction/output helper arrays
  {
    bool _t1_pass = (size_res > 0 &&
                     size_max > 0 &&
                     res != NULL &&
                     max_u_arr != NULL &&
                     max_v_arr != NULL &&
                     pres_sum != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_host_ptrs_aux_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // set initial BCs
  set_BCs_host (u, v);

  Real max_u = SMALL;
  Real max_v = SMALL;
  // get max velocity for initial values (including BCs)
  #pragma unroll
  for (int col = 0; col < NUM + 2; ++col) {
    #pragma unroll
    for (int row = 1; row < NUM + 2; ++row) {
      max_u = fmax(max_u, fabs( u(col, row) ));
    }
  }

  #pragma unroll
  for (int col = 1; col < NUM + 2; ++col) {
    #pragma unroll
    for (int row = 0; row < NUM + 2; ++row) {
      max_v = fmax(max_v, fabs( v(col, row) ));
    }
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Initial host field numerical validity before host-to-device transfer
  {
    bool _t1_pass = true;
    for (int i = 0; i < size; ++i) {
      if (!isfinite((double)F[i]) || !isfinite((double)u[i]) ||
          !isfinite((double)G[i]) || !isfinite((double)v[i])) { _t1_pass = false; break; }
    }
    if (_t1_pass) {
      for (int i = 0; i < size_pres; ++i) {
        if (!isfinite((double)pres_red[i]) || !isfinite((double)pres_black[i])) { _t1_pass = false; break; }
      }
    }
    if (_t1_pass) {
      _t1_pass = (isfinite((double)max_u) && isfinite((double)max_v) && max_u > ZERO && max_v > ZERO);
    }
    printf("LASSI-VERIFY ASSERTIONS [main_input_finite_before_htod]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // allocate and transfer device memory
  Real* u_d;
  Real* F_d;
  Real* v_d;
  Real* G_d;

  Real* pres_red_d;
  Real* pres_black_d;
  Real* pres_sum_d;
  Real* res_d;

  Real* max_u_d;
  Real* max_v_d;

  cudaMalloc ((void**) &u_d, size * sizeof(Real));
  cudaMalloc ((void**) &F_d, size * sizeof(Real));
  cudaMalloc ((void**) &v_d, size * sizeof(Real));
  cudaMalloc ((void**) &G_d, size * sizeof(Real));

  cudaMalloc ((void**) &pres_red_d, size_pres * sizeof(Real));
  cudaMalloc ((void**) &pres_black_d, size_pres * sizeof(Real));

  cudaMalloc ((void**) &pres_sum_d, size_res * sizeof(Real));
  cudaMalloc ((void**) &res_d, size_res * sizeof(Real));
  cudaMalloc ((void**) &max_u_d, size_max * sizeof(Real));
  cudaMalloc ((void**) &max_v_d, size_max * sizeof(Real));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Device allocation pointer validity prior to memory transfers
  {
    bool _t1_pass = (u_d != NULL && F_d != NULL && v_d != NULL && G_d != NULL &&
                     pres_red_d != NULL && pres_black_d != NULL &&
                     pres_sum_d != NULL && res_d != NULL &&
                     max_u_d != NULL && max_v_d != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_device_ptrs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // copy to device memory
  cudaMemcpy (u_d, u, size * sizeof(Real), cudaMemcpyHostToDevice);
  cudaMemcpy (F_d, F, size * sizeof(Real), cudaMemcpyHostToDevice);
  cudaMemcpy (v_d, v, size * sizeof(Real), cudaMemcpyHostToDevice);
  cudaMemcpy (G_d, G, size * sizeof(Real), cudaMemcpyHostToDevice);
  cudaMemcpy (pres_red_d, pres_red, size_pres * sizeof(Real), cudaMemcpyHostToDevice);
  cudaMemcpy (pres_black_d, pres_black, size_pres * sizeof(Real), cudaMemcpyHostToDevice);

  Real time = time_start;

  // time-step size based on grid and Reynolds number
  Real dt_Re = 0.5 * Re_num / ((1.0 / (dx * dx)) + (1.0 / (dy * dy)));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Time-step stability scalar validity before solver loop
  {
    bool _t1_pass = (isfinite((double)time) &&
                     isfinite((double)dt_Re) &&
                     dt_Re > ZERO &&
                     time == time_start);
    printf("LASSI-VERIFY ASSERTIONS [main_preloop_timestep_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaDeviceSynchronize();
  auto start = std::chrono::steady_clock::now();

  // time iteration loop
  while (time < time_end) {

    // calculate time step based on stability and CFL
    dt = fmin((dx / max_u), (dy / max_v));
    dt = tau * fmin(dt_Re, dt);

    if ((time + dt) >= time_end) {
      dt = time_end - time;
    }

    // calculate F and G    
    calculate_F <<<grid_F, block_F>>> (dt, u_d, v_d, F_d);
    calculate_G <<<grid_G, block_G>>> (dt, u_d, v_d, G_d);

    // get L2 norm of initial pressure
    sum_pressure <<<grid_pr, block_pr>>> (pres_red_d, pres_black_d, pres_sum_d);
    cudaMemcpy (pres_sum, pres_sum_d, size_res * sizeof(Real), cudaMemcpyDeviceToHost);

    Real p0_norm = ZERO;
    #pragma unroll
    for (int i = 0; i < size_res; ++i) {
      p0_norm += pres_sum[i];
    }

    p0_norm = sqrt(p0_norm / ((Real)(NUM * NUM)));
    if (p0_norm < 0.0001) {
      p0_norm = 1.0;
    }

    // ensure all kernels are finished
    //cudaDeviceSynchronize();

    Real norm_L2;

    // calculate new pressure
    // red-black Gauss-Seidel with SOR iteration loop
    for (iter = 1; iter <= it_max; ++iter) {

      // set pressure boundary conditions
      set_horz_pres_BCs <<<grid_hpbc, block_hpbc>>> (pres_red_d, pres_black_d);
      set_vert_pres_BCs <<<grid_vpbc, block_vpbc>>> (pres_red_d, pres_black_d);

      // ensure kernel finished
      //cudaDeviceSynchronize();

      // update red cells
      red_kernel <<<grid_pr, block_pr>>> (dt, F_d, G_d, pres_black_d, pres_red_d);

      // ensure red kernel finished
      //cudaDeviceSynchronize();

      // update black cells
      black_kernel <<<grid_pr, block_pr>>> (dt, F_d, G_d, pres_red_d, pres_black_d);

      // ensure red kernel finished
      //cudaDeviceSynchronize();

      // calculate residual values
      calc_residual <<<grid_pr, block_pr>>> (dt, F_d, G_d, pres_red_d, pres_black_d, res_d);

      // transfer residual value(s) back to CPU
      cudaMemcpy (res, res_d, size_res * sizeof(Real), cudaMemcpyDeviceToHost);

      norm_L2 = ZERO;
      #pragma unroll
      for (int i = 0; i < size_res; ++i) {
        norm_L2 += res[i];
      }

      // calculate residual
      norm_L2 = sqrt(norm_L2 / ((Real)(NUM * NUM))) / p0_norm;

      // if tolerance has been reached, end SOR iterations
      if (norm_L2 < tol) {
        break;
      }  
    } // end for

    printf("Time = %f, delt = %e, iter = %i, res = %e\n", time + dt, dt, iter, norm_L2);

    // calculate new velocities and transfer maximums back

    calculate_u <<<grid_pr, block_pr>>> (dt, F_d, pres_red_d, pres_black_d, u_d, max_u_d);
    cudaMemcpy (max_u_arr, max_u_d, size_max * sizeof(Real), cudaMemcpyDeviceToHost);

    calculate_v <<<grid_pr, block_pr>>> (dt, G_d, pres_red_d, pres_black_d, v_d, max_v_d);
    cudaMemcpy (max_v_arr, max_v_d, size_max * sizeof(Real), cudaMemcpyDeviceToHost);

    // get maximum u- and v- velocities
    max_v = SMALL;
    max_u = SMALL;

    #pragma unroll
    for (int i = 0; i < size_max; ++i) {
      Real test_u = max_u_arr[i];
      max_u = fmax(max_u, test_u);

      Real test_v = max_v_arr[i];
      max_v = fmax(max_v, test_v);
    }

    // set velocity boundary conditions
    set_BCs <<<grid_bcs, block_bcs>>> (u_d, v_d);

    // increase time
    time += dt;

    // single time step
    //break;

  } // end while

  cudaDeviceSynchronize();
  auto end = std::chrono::steady_clock::now();
  auto elapsed_time = std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count();
  printf("\nTotal execution time of the iteration loop: %f (s)\n", elapsed_time * 1e-9f);

  // transfer final temperature values back
  cudaMemcpy (u, u_d, size * sizeof(Real), cudaMemcpyDeviceToHost);
  cudaMemcpy (v, v_d, size * sizeof(Real), cudaMemcpyDeviceToHost);
  cudaMemcpy (pres_red, pres_red_d, size_pres * sizeof(Real), cudaMemcpyDeviceToHost);
  cudaMemcpy (pres_black, pres_black_d, size_pres * sizeof(Real), cudaMemcpyDeviceToHost);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Final host arrays finite after device-to-host transfers
  {
    bool _t1_pass = true;
    for (int i = 0; i < size; ++i) {
      if (!isfinite((double)u[i]) || !isfinite((double)v[i])) { _t1_pass = false; break; }
    }
    if (_t1_pass) {
      for (int i = 0; i < size_pres; ++i) {
        if (!isfinite((double)pres_red[i]) || !isfinite((double)pres_black[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_output_finite_after_dtoh]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Final scalar outputs and solver state validity at function exit
  {
    bool _t1_pass = (isfinite((double)time) &&
                     isfinite((double)dt) &&
                     isfinite((double)max_u) &&
                     isfinite((double)max_v) &&
                     isfinite((double)elapsed_time) &&
                     max_u >= ZERO &&
                     max_v >= ZERO &&
                     dt >= ZERO &&
                     time >= time_start &&
                     time <= (time_end + (Real)1e-12) &&
                     iter >= 1 &&
                     iter <= it_max);
    printf("LASSI-VERIFY ASSERTIONS [main_exit_scalars_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // write data to file
  FILE * pfile;
  pfile = fopen("velocity_gpu.dat", "w");
  fprintf(pfile, "#x\ty\tu\tv\n");
  if (pfile != NULL) {
    for (int row = 0; row < NUM; ++row) {
      for (int col = 0; col < NUM; ++col) {

        Real u_ij = u[(col * NUM) + row];
        Real u_im1j;
        if (col == 0) {
          u_im1j = 0.0;
        } else {
          u_im1j = u[(col - 1) * NUM + row];
        }

        u_ij = (u_ij + u_im1j) / 2.0;

        Real v_ij = v[(col * NUM) + row];
        Real v_ijm1;
        if (row == 0) {
          v_ijm1 = 0.0;
        } else {
          v_ijm1 = v[(col * NUM) + row - 1];
        }

        v_ij = (v_ij + v_ijm1) / 2.0;

        fprintf(pfile, "%f\t%f\t%f\t%f\n", ((Real)col + 0.5) * dx, ((Real)row + 0.5) * dy, u_ij, v_ij);
      }
    }
  }

  fclose(pfile);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Output file handle lifecycle and final field finiteness before return
  {
    bool _t1_pass = true;
    for (int row = 0; row < NUM && _t1_pass; ++row) {
      for (int col = 0; col < NUM; ++col) {
        Real _t1_uij = u[(col * NUM) + row];
        Real _t1_uim1j = (col == 0) ? (Real)0.0 : u[(col - 1) * NUM + row];
        Real _t1_vij = v[(col * NUM) + row];
        Real _t1_vijm1 = (row == 0) ? (Real)0.0 : v[(col * NUM) + row - 1];
        Real _t1_uc = (_t1_uij + _t1_uim1j) / (Real)2.0;
        Real _t1_vc = (_t1_vij + _t1_vijm1) / (Real)2.0;
        if (!isfinite((double)_t1_uc) || !isfinite((double)_t1_vc)) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_final_centered_velocity_finite]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // free device memory
  cudaFree(u_d);
  cudaFree(v_d);
  cudaFree(F_d);
  cudaFree(G_d);
  cudaFree(pres_red_d);
  cudaFree(pres_black_d);
  cudaFree(max_u_d);
  cudaFree(max_v_d);
  cudaFree(pres_sum_d);
  cudaFree(res_d);

  free(pres_red);
  free(pres_black);
  free(u);
  free(v);
  free(F);
  free(G);
  free(max_u_arr);
  free(max_v_arr);
  free(res);
  free(pres_sum);

  return 0;
}
```


----- STARTING CODE INSERT - Passthrough Data Dump Blocks -----


=====> VERIFY THE PROMPTING <=====

TEMPERATURE: 0.2

SYSTEM: You are an expert C++/CUDA engineer. Your goal is to refactor provided CUDA C++ code to reduce wall-clock execution time by improving GPU and CPU utilization and parallelism, while preserving all original functionality and expected output. You always surround your generated code with the three characters ```.

PROMPT: 
        You are preparing GPU-accelerated scientific code for Tier 2 empirical equivalence checking
        as part of the LASSI-Verify verification framework.

        Your task is to instrument the source code with **passthrough data blocks** at key observation
        points. These blocks capture computational state to stdout in a structured, labeled format.
        Both the source (SOURCE_verify) and every LASSI-generated refactored version will carry these
        blocks unaltered, enabling deterministic comparison of observable output at 

```cpp
/** GPU solver for 2D lid-driven cavity problem, using finite difference method
 * \file main_gpu.cu
 *
 * \author Kyle E. Niemeyer
 * \date 09/27/2012
 *
 * Solve the incompressible, isothermal 2D Navier–Stokes equations for a square
 * lid-driven cavity on a GPU (via CUDA), using the finite difference method.
 * To change the grid resolution, modify "NUM". In addition, the problem is controlled
 * by the Reynolds number ("Re_num").
 * 
 * Based on the methodology given in Chapter 3 of "Numerical Simulation in Fluid
 * Dynamics", by M. Griebel, T. Dornseifer, and T. Neunhoeffer. SIAM, Philadelphia,
 * PA, 1998.
 * 
 * Boundary conditions:
 * u = 0 and v = 0 at x = 0, x = L, y = 0
 * u = ustar at y = H
 * v = 0 at y = H
 */

#include <stdlib.h>
#include <stdio.h>
#include <math.h>
#include <chrono>
#include <cuda.h>
#include <cstdio>
#include <cmath>

/** Problem size along one side; total number of cells is this squared */
#define NUM 512

// block size
#define BLOCK_SIZE 128

/** Double precision */
#define DOUBLE

#ifdef DOUBLE
#define Real double

#define ZERO 0.0
#define ONE 1.0
#define TWO 2.0
#define FOUR 4.0

#define SMALL 1.0e-10;

/** Reynolds number */
const Real Re_num = 1000.0;

/** SOR relaxation parameter */
const Real omega = 1.7;

/** Discretization mixture parameter (gamma) */
const Real mix_param = 0.9;

/** Safety factor for time step modification */
const Real tau = 0.5;

/** Body forces in x- and y- directions */
const Real gx = 0.0;
const Real gy = 0.0;

/** Domain size (non-dimensional) */
#define xLength 1.0
#define yLength 1.0
#else
#define Real float

// replace double functions with float versions
#undef fmin
#define fmin fminf
#undef fmax
#define fmax fmaxf
#undef fabs
#define fabs fabsf
#undef sqrt
#define sqrt sqrtf

#define ZERO 0.0f
#define ONE 1.0f
#define TWO 2.0f
#define FOUR 4.0f
#define SMALL 1.0e-10f;

/** Reynolds number */
const Real Re_num = 1000.0f;

/** SOR relaxation parameter */
const Real omega = 1.7f;

/** Discretization mixture parameter (gamma) */
const Real mix_param = 0.9f;

/** Safety factor for time step modification */
const Real tau = 0.5f;

/** Body forces in x- and y- directions */
const Real gx = 0.0f;
const Real gy = 0.0f;

/** Domain size (non-dimensional) */
#define xLength 1.0f
#define yLength 1.0f
#endif

/** Mesh sizes */
const Real dx = xLength / NUM;
const Real dy = yLength / NUM;

/** Max macro (type safe, from GNU) */
//#define MAX(a,b) ({ __typeof__ (a) _a = (a); __typeof__ (b) _b = (b); _a > _b ? _a : _b; })

/** Min macro (type safe) */
//#define MIN(a,b) ({ __typeof__ (a) _a = (a); __typeof__ (b) _b = (b); _a < _b ? _a : _b; })

// map two-dimensional indices to one-dimensional memory
#define u(I, J) u[((I) * ((NUM) + 2)) + (J)]
#define v(I, J) v[((I) * ((NUM) + 2)) + (J)]
#define F(I, J) F[((I) * ((NUM) + 2)) + (J)]
#define G(I, J) G[((I) * ((NUM) + 2)) + (J)]
#define pres_red(I, J) pres_red[((I) * ((NUM_2) + 2)) + (J)]
#define pres_black(I, J) pres_black[((I) * ((NUM_2) + 2)) + (J)]

///////////////////////////////////////////////////////////////////////////////
__host__
void set_BCs_host (Real* u, Real* v) 
{
  int ind;

  // loop through rows and columns
  for (ind = 0; ind < NUM + 2; ++ind) {

    // left boundary
    u(0, ind) = ZERO;
    v(0, ind) = -v(1, ind);

    // right boundary
    u(NUM, ind) = ZERO;
    v(NUM + 1, ind) = -v(NUM, ind);

    // bottom boundary
    u(ind, 0) = -u(ind, 1);
    v(ind, 0) = ZERO;

    // top boundary
    u(ind, NUM + 1) = TWO - u(ind, NUM);
    v(ind, NUM) = ZERO;

    if (ind == NUM) {
      // left boundary
      u(0, 0) = ZERO;
      v(0, 0) = -v(1, 0);
      u(0, NUM + 1) = ZERO;
      v(0, NUM + 1) = -v(1, NUM + 1);

      // right boundary
      u(NUM, 0) = ZERO;
      v(NUM + 1, 0) = -v(NUM, 0);
      u(NUM, NUM + 1) = ZERO;
      v(NUM + 1, NUM + 1) = -v(NUM, NUM + 1);

      // bottom boundary
      u(0, 0) = -u(0, 1);
      v(0, 0) = ZERO;
      u(NUM + 1, 0) = -u(NUM + 1, 1);
      v(NUM + 1, 0) = ZERO;

      // top boundary
      u(0, NUM + 1) = TWO - u(0, NUM);
      v(0, NUM) = ZERO;
      u(NUM + 1, NUM + 1) = TWO - u(NUM + 1, NUM);
      v(ind, NUM + 1) = ZERO;
    } // end if

  } // end for

} // end set_BCs_host

///////////////////////////////////////////////////////////////////////////////
__global__
void set_BCs (Real*__restrict__ u, Real*__restrict__ v) 
{
  int ind = (blockIdx.x * blockDim.x) + threadIdx.x + 1;

  // left boundary
  u(0, ind) = ZERO;
  v(0, ind) = -v(1, ind);

  // right boundary
  u(NUM, ind) = ZERO;
  v(NUM + 1, ind) = -v(NUM, ind);

  // bottom boundary
  u(ind, 0) = -u(ind, 1);
  v(ind, 0) = ZERO;

  // top boundary
  u(ind, NUM + 1) = TWO - u(ind, NUM);
  v(ind, NUM) = ZERO;

  if (ind == NUM) {
    // left boundary
    u(0, 0) = ZERO;
    v(0, 0) = -v(1, 0);
    u(0, NUM + 1) = ZERO;
    v(0, NUM + 1) = -v(1, NUM + 1);

    // right boundary
    u(NUM, 0) = ZERO;
    v(NUM + 1, 0) = -v(NUM, 0);
    u(NUM, NUM + 1) = ZERO;
    v(NUM + 1, NUM + 1) = -v(NUM, NUM + 1);

    // bottom boundary
    u(0, 0) = -u(0, 1);
    v(0, 0) = ZERO;
    u(NUM + 1, 0) = -u(NUM + 1, 1);
    v(NUM + 1, 0) = ZERO;

    // top boundary
    u(0, NUM + 1) = TWO - u(0, NUM);
    v(0, NUM) = ZERO;
    u(NUM + 1, NUM + 1) = TWO - u(NUM + 1, NUM);
    v(ind, NUM + 1) = ZERO;
  } // end if

} // end set_BCs

///////////////////////////////////////////////////////////////////////////////

__global__ 
void calculate_F (const Real dt,
                  const Real*__restrict__ u,
                  const Real*__restrict__ v,
                        Real*__restrict__ F) 
{  
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  if (col == NUM) {
    // right boundary, F_ij = u_ij
    // also do left boundary
    F(0, row) = u(0, row);
    F(NUM, row) = u(NUM, row);
  } else {

    // u velocities
    Real u_ij = u(col, row);
    Real u_ip1j = u(col + 1, row);
    Real u_ijp1 = u(col, row + 1);
    Real u_im1j = u(col - 1, row);
    Real u_ijm1 = u(col, row - 1);

    // v velocities
    Real v_ij = v(col, row);
    Real v_ip1j = v(col + 1, row);
    Real v_ijm1 = v(col, row - 1);
    Real v_ip1jm1 = v(col + 1, row - 1);

    // finite differences
    Real du2dx, duvdy, d2udx2, d2udy2;

    du2dx = (((u_ij + u_ip1j) * (u_ij + u_ip1j) - (u_im1j + u_ij) * (u_im1j + u_ij))
        + mix_param * (fabs(u_ij + u_ip1j) * (u_ij - u_ip1j)
          - fabs(u_im1j + u_ij) * (u_im1j - u_ij))) / (FOUR * dx);
    duvdy = ((v_ij + v_ip1j) * (u_ij + u_ijp1) - (v_ijm1 + v_ip1jm1) * (u_ijm1 + u_ij)
        + mix_param * (fabs(v_ij + v_ip1j) * (u_ij - u_ijp1)
          - fabs(v_ijm1 + v_ip1jm1) * (u_ijm1 - u_ij))) / (FOUR * dy);
    d2udx2 = (u_ip1j - (TWO * u_ij) + u_im1j) / (dx * dx);
    d2udy2 = (u_ijp1 - (TWO * u_ij) + u_ijm1) / (dy * dy);

    F(col, row) = u_ij + dt * (((d2udx2 + d2udy2) / Re_num) - du2dx - duvdy + gx);

  } // end if

} // end calculate_F

///////////////////////////////////////////////////////////////////////////////

__global__ 
void calculate_G (const Real dt,
                  const Real*__restrict__ u,
                  const Real*__restrict__ v,
                        Real*__restrict__ G) 
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  if (row == NUM) {
    // top and bottom boundaries
    G(col, 0) = v(col, 0);
    G(col, NUM) = v(col, NUM);

  } else {

    // u velocities
    Real u_ij = u(col, row);
    Real u_ijp1 = u(col, row + 1);
    Real u_im1j = u(col - 1, row);
    Real u_im1jp1 = u(col - 1, row + 1);

    // v velocities
    Real v_ij = v(col, row);
    Real v_ijp1 = v(col, row + 1);
    Real v_ip1j = v(col + 1, row);
    Real v_ijm1 = v(col, row - 1);
    Real v_im1j = v(col - 1, row);

    // finite differences
    Real dv2dy, duvdx, d2vdx2, d2vdy2;

    dv2dy = ((v_ij + v_ijp1) * (v_ij + v_ijp1) - (v_ijm1 + v_ij) * (v_ijm1 + v_ij)
        + mix_param * (fabs(v_ij + v_ijp1) * (v_ij - v_ijp1)
          - fabs(v_ijm1 + v_ij) * (v_ijm1 - v_ij))) / (FOUR * dy);
    duvdx = ((u_ij + u_ijp1) * (v_ij + v_ip1j) - (u_im1j + u_im1jp1) * (v_im1j + v_ij)
        + mix_param * (fabs(u_ij + u_ijp1) * (v_ij - v_ip1j) 
          - fabs(u_im1j + u_im1jp1) * (v_im1j - v_ij))) / (FOUR * dx);
    d2vdx2 = (v_ip1j - (TWO * v_ij) + v_im1j) / (dx * dx);
    d2vdy2 = (v_ijp1 - (TWO * v_ij) + v_ijm1) / (dy * dy);

    G(col, row) = v_ij + dt * (((d2vdx2 + d2vdy2) / Re_num) - dv2dy - duvdx + gy);

  } // end if

} // end calculate_G

///////////////////////////////////////////////////////////////////////////////

__global__ 
void sum_pressure (const Real*__restrict__ pres_red,
                   const Real*__restrict__ pres_black, 
                         Real*__restrict__ pres_sum) 
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  // shared memory for block's sum
  __shared__ Real sum_cache[BLOCK_SIZE];

  int NUM_2 = NUM >> 1;

  Real pres_r = pres_red(col, row);
  Real pres_b = pres_black(col, row);

  // add squared pressure
  sum_cache[threadIdx.x] = (pres_r * pres_r) + (pres_b * pres_b);

  // synchronize threads in block to ensure all thread values stored
  __syncthreads();

  // add up values for block
  int i = BLOCK_SIZE >> 1;
  while (i != 0) {
    if (threadIdx.x < i) {
      sum_cache[threadIdx.x] += sum_cache[threadIdx.x + i];
    }
    __syncthreads();
    i >>= 1;
  }

  // store block's summed values
  if (threadIdx.x == 0) {
    pres_sum[blockIdx.y + (gridDim.y * blockIdx.x)] = sum_cache[0];
  }

} // end sum_pressure

///////////////////////////////////////////////////////////////////////////////
__global__ 
void set_horz_pres_BCs (Real*__restrict__ pres_red, Real*__restrict__ pres_black) 
{
  int col = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  col = (col * 2) - 1;

  int NUM_2 = NUM >> 1;

  // p_i,0 = p_i,1
  pres_black(col, 0) = pres_red(col, 1);
  pres_red(col + 1, 0) = pres_black(col + 1, 1);

  // p_i,jmax+1 = p_i,jmax
  pres_red(col, NUM_2 + 1) = pres_black(col, NUM_2);
  pres_black(col + 1, NUM_2 + 1) = pres_red(col + 1, NUM_2);

} // end set_horz_pres_BCs

//////////////////////////////////////////////////////////////////////////////

__global__
void set_vert_pres_BCs (Real*__restrict__ pres_red, Real*__restrict__ pres_black) 
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;

  int NUM_2 = NUM >> 1;

  // p_0,j = p_1,j
  pres_black(0, row) = pres_red(1, row);
  pres_red(0, row) = pres_black(1, row);

  // p_imax+1,j = p_imax,j
  pres_black(NUM + 1, row) = pres_red(NUM, row);
  pres_red(NUM + 1, row) = pres_black(NUM, row);

} // end set_pressure_BCs

///////////////////////////////////////////////////////////////////////////////

/** Function to update pressure for red cells
 * 
 * \param[in]    dt      time-step size
 * \param[in]    F      array of discretized x-momentum eqn terms
 * \param[in]    G      array of discretized y-momentum eqn terms
 * \param[in]    pres_black  pressure values of black cells
 * \param[inout]  pres_red  pressure values of red cells
 */
__global__
void red_kernel (const Real dt,
                 const Real*__restrict__ F, 
                 const Real*__restrict__ G,
                 const Real*__restrict__ pres_black,
                       Real*__restrict__ pres_red) 
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  int NUM_2 = NUM >> 1;      

  Real p_ij = pres_red(col, row);

  Real p_im1j = pres_black(col - 1, row);
  Real p_ip1j = pres_black(col + 1, row);
  Real p_ijm1 = pres_black(col, row - (col & 1));
  Real p_ijp1 = pres_black(col, row + ((col + 1) & 1));

  // right-hand side
  Real rhs = (((F(col, (2 * row) - (col & 1))
          - F(col - 1, (2 * row) - (col & 1))) / dx)
      + ((G(col, (2 * row) - (col & 1))
          - G(col, (2 * row) - (col & 1) - 1)) / dy)) / dt;

  pres_red(col, row) = p_ij * (ONE - omega) + omega * 
    (((p_ip1j + p_im1j) / (dx * dx)) + ((p_ijp1 + p_ijm1) / (dy * dy)) - 
     rhs) / ((TWO / (dx * dx)) + (TWO / (dy * dy)));

} // end red_kernel

///////////////////////////////////////////////////////////////////////////////

/** Function to update pressure for black cells
 * 
 * \param[in]    dt      time-step size
 * \param[in]    F      array of discretized x-momentum eqn terms
 * \param[in]    G      array of discretized y-momentum eqn terms
 * \param[in]    pres_red  pressure values of red cells
 * \param[inout]  pres_black  pressure values of black cells
 */
__global__ 
void black_kernel (const Real dt,
                   const Real*__restrict__ F, 
                   const Real*__restrict__ G,
                   const Real*__restrict__ pres_red, 
                         Real*__restrict__ pres_black) 
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  int NUM_2 = NUM >> 1;

  Real p_ij = pres_black(col, row);

  Real p_im1j = pres_red(col - 1, row);
  Real p_ip1j = pres_red(col + 1, row);
  Real p_ijm1 = pres_red(col, row - ((col + 1) & 1));
  Real p_ijp1 = pres_red(col, row + (col & 1));

  // right-hand side
  Real rhs = (((F(col, (2 * row) - ((col + 1) & 1))
          - F(col - 1, (2 * row) - ((col + 1) & 1))) / dx)
      + ((G(col, (2 * row) - ((col + 1) & 1))
          - G(col, (2 * row) - ((col + 1) & 1) - 1)) / dy)) / dt;

  pres_black(col, row) = p_ij * (ONE - omega) + omega * 
    (((p_ip1j + p_im1j) / (dx * dx)) + ((p_ijp1 + p_ijm1) / (dy * dy)) - 
     rhs) / ((TWO / (dx * dx)) + (TWO / (dy * dy)));

} // end black_kernel

///////////////////////////////////////////////////////////////////////////////

__global__
void calc_residual (const Real dt,
                    const Real*__restrict__ F,
                    const Real*__restrict__ G, 
                    const Real*__restrict__ pres_red,
                    const Real*__restrict__ pres_black,
                          Real*__restrict__ res_array)
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  int NUM_2 = NUM >> 1;

  Real p_ij, p_im1j, p_ip1j, p_ijm1, p_ijp1, rhs, res, res2;

  // red point
  p_ij = pres_red(col, row);

  p_im1j = pres_black(col - 1, row);
  p_ip1j = pres_black(col + 1, row);
  p_ijm1 = pres_black(col, row - (col & 1));
  p_ijp1 = pres_black(col, row + ((col + 1) & 1));

  rhs = (((F(col, (2 * row) - (col & 1)) - F(col - 1, (2 * row) - (col & 1))) / dx)
      +  ((G(col, (2 * row) - (col & 1)) - G(col, (2 * row) - (col & 1) - 1)) / dy)) / dt;

  // calculate residual
  res = ((p_ip1j - (TWO * p_ij) + p_im1j) / (dx * dx))
    + ((p_ijp1 - (TWO * p_ij) + p_ijm1) / (dy * dy)) - rhs;

  // black point
  p_ij = pres_black(col, row);

  p_im1j = pres_red(col - 1, row);
  p_ip1j = pres_red(col + 1, row);
  p_ijm1 = pres_red(col, row - ((col + 1) & 1));
  p_ijp1 = pres_red(col, row + (col & 1));

  // right-hand side
  rhs = (((F(col, (2 * row) - ((col + 1) & 1)) - F(col - 1, (2 * row) - ((col + 1) & 1))) / dx)
      +  ((G(col, (2 * row) - ((col + 1) & 1)) - G(col, (2 * row) - ((col + 1) & 1) - 1)) / dy)) / dt;

  // calculate residual
  res2 = ((p_ip1j - (TWO * p_ij) + p_im1j) / (dx * dx))
    + ((p_ijp1 - (TWO * p_ij) + p_ijm1) / (dy * dy)) - rhs;

  // shared memory for block's sum
  __shared__ Real sum_cache[BLOCK_SIZE];

  sum_cache[threadIdx.x] = (res * res) + (res2 * res2);

  // synchronize threads in block to ensure all residuals stored
  __syncthreads();

  // add up squared residuals for block
  int i = BLOCK_SIZE >> 1;
  while (i != 0) {
    if (threadIdx.x < i) {
      sum_cache[threadIdx.x] += sum_cache[threadIdx.x + i];
    }
    __syncthreads();
    i >>= 1;
  }

  // store block's summed residuals
  if (threadIdx.x == 0) {
    res_array[blockIdx.y + (gridDim.y * blockIdx.x)] = sum_cache[0];
  }
} 

///////////////////////////////////////////////////////////////////////////////

__global__ 
void calculate_u (const Real dt,
                  const Real*__restrict__ F, 
                  const Real*__restrict__ pres_red,
                  const Real*__restrict__ pres_black, 
                        Real*__restrict__ u,
                        Real*__restrict__ max_u)
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  // allocate shared memory to store max velocities
  __shared__ Real max_cache[BLOCK_SIZE];
  max_cache[threadIdx.x] = ZERO;

  int NUM_2 = NUM >> 1;
  Real new_u = ZERO;

  if (col != NUM) {

    Real p_ij, p_ip1j, new_u2;

    // red point
    p_ij = pres_red(col, row);
    p_ip1j = pres_black(col + 1, row);

    new_u = F(col, (2 * row) - (col & 1)) - (dt * (p_ip1j - p_ij) / dx);
    u(col, (2 * row) - (col & 1)) = new_u;

    // black point
    p_ij = pres_black(col, row);
    p_ip1j = pres_red(col + 1, row);

    new_u2 = F(col, (2 * row) - ((col + 1) & 1)) - (dt * (p_ip1j - p_ij) / dx);
    u(col, (2 * row) - ((col + 1) & 1)) = new_u2;

    // check for max of these two
    new_u = fmax(fabs(new_u), fabs(new_u2));

    if ((2 * row) == NUM) {
      // also test for max velocity at vertical boundary
      new_u = fmax(new_u, fabs( u(col, NUM + 1) ));
    }
  } else {
    // check for maximum velocity in boundary cells also
    new_u = fmax(fabs( u(NUM, (2 * row)) ), fabs( u(0, (2 * row)) ));
    new_u = fmax(fabs( u(NUM, (2 * row) - 1) ), new_u);
    new_u = fmax(fabs( u(0, (2 * row) - 1) ), new_u);

    new_u = fmax(fabs( u(NUM + 1, (2 * row)) ), new_u);
    new_u = fmax(fabs( u(NUM + 1, (2 * row) - 1) ), new_u);

  } // end if

  // store maximum u for block from each thread
  max_cache[threadIdx.x] = new_u;

  // synchronize threads in block to ensure all velocities stored
  __syncthreads();

  // calculate maximum for block
  int i = BLOCK_SIZE >> 1;
  while (i != 0) {
    if (threadIdx.x < i) {
      max_cache[threadIdx.x] = fmax(max_cache[threadIdx.x], max_cache[threadIdx.x + i]);
    }
    __syncthreads();
    i >>= 1;
  }

  // store block's maximum
  if (threadIdx.x == 0) {
    max_u[blockIdx.y + (gridDim.y * blockIdx.x)] = max_cache[0];
  }
} // end calculate_u

///////////////////////////////////////////////////////////////////////////////

__global__ 
void calculate_v (const Real dt,
                  const Real*__restrict__ G, 
                  const Real*__restrict__ pres_red,
                  const Real*__restrict__ pres_black, 
                        Real*__restrict__ v,
                        Real*__restrict__ max_v)
{
  int row = (blockIdx.x * blockDim.x) + threadIdx.x + 1;
  int col = (blockIdx.y * blockDim.y) + threadIdx.y + 1;

  // allocate shared memory to store maximum velocities
  __shared__ Real max_cache[BLOCK_SIZE];
  max_cache[threadIdx.x] = ZERO;

  int NUM_2 = NUM >> 1;
  Real new_v = ZERO;

  if (row != NUM_2) {
    Real p_ij, p_ijp1, new_v2;

    // red pressure point
    p_ij = pres_red(col, row);
    p_ijp1 = pres_black(col, row + ((col + 1) & 1));

    new_v = G(col, (2 * row) - (col & 1)) - (dt * (p_ijp1 - p_ij) / dy);
    v(col, (2 * row) - (col & 1)) = new_v;

    // black pressure point
    p_ij = pres_black(col, row);
    p_ijp1 = pres_red(col, row + (col & 1));

    new_v2 = G(col, (2 * row) - ((col + 1) & 1)) - (dt * (p_ijp1 - p_ij) / dy);
    v(col, (2 * row) - ((col + 1) & 1)) = new_v2;

    // check for max of these two
    new_v = fmax(fabs(new_v), fabs(new_v2));

    if (col == NUM) {
      // also test for max velocity at vertical boundary
      new_v = fmax(new_v, fabs( v(NUM + 1, (2 * row)) ));
    }

  } else {

    if ((col & 1) == 1) {
      // black point is on boundary, only calculate red point below it
      Real p_ij = pres_red(col, row);
      Real p_ijp1 = pres_black(col, row + ((col + 1) & 1));

      new_v = G(col, (2 * row) - (col & 1)) - (dt * (p_ijp1 - p_ij) / dy);
      v(col, (2 * row) - (col & 1)) = new_v;
    } else {
      // red point is on boundary, only calculate black point below it
      Real p_ij = pres_black(col, row);
      Real p_ijp1 = pres_red(col, row + (col & 1));

      new_v = G(col, (2 * row) - ((col + 1) & 1)) - (dt * (p_ijp1 - p_ij) / dy);
      v(col, (2 * row) - ((col + 1) & 1)) = new_v;
    }

    // get maximum v velocity
    new_v = fabs(new_v);

    // check for maximum velocity in boundary cells also
    new_v = fmax(fabs( v(col, NUM) ), new_v);
    new_v = fmax(fabs( v(col, 0) ), new_v);

    new_v = fmax(fabs( v(col, NUM + 1) ), new_v);
  } // end if

  // store absolute value of velocity
  max_cache[threadIdx.x] = new_v;

  // synchronize threads in block to ensure all velocities stored
  __syncthreads();

  // calculate maximum for block
  int i = BLOCK_SIZE >> 1;
  while (i != 0) {
    if (threadIdx.x < i) {
      max_cache[threadIdx.x] = fmax(max_cache[threadIdx.x], max_cache[threadIdx.x + i]);
    }
    __syncthreads();
    i >>= 1;
  }

  // store block's summed residuals
  if (threadIdx.x == 0) {
    max_v[blockIdx.y + (gridDim.y * blockIdx.x)] = max_cache[0];
  }
} // end calculate_v

///////////////////////////////////////////////////////////////////////////////

int main (int argc, char *argv[])
{
  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Entry parameter and compile-time configuration validity
  {
    bool _t1_pass = (NUM > 0 &&
                     BLOCK_SIZE > 0 &&
                     (NUM % BLOCK_SIZE) == 0 &&
                     ((NUM / 2) % BLOCK_SIZE) == 0 &&
                     Re_num > ZERO &&
                     omega > ZERO &&
                     tau > ZERO &&
                     tau <= ONE);
    printf("LASSI-VERIFY ASSERTIONS [main_entry_config_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // iterations for Red-Black Gauss-Seidel with SOR
  int iter = 0;
  const int it_max = 1000000;

  // SOR iteration tolerance
  const Real tol = 0.001;

  // time range
  const Real time_start = 0.0;
  const Real time_end = 0.001; //20.0;

  // initial time step size
  Real dt = 0.02;

  int size = (NUM + 2) * (NUM + 2);
  int size_pres = ((NUM / 2) + 2) * (NUM + 2);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Core scalar dimensions and time configuration validity
  {
    bool _t1_pass = (size > 0 &&
                     size_pres > 0 &&
                     it_max > 0 &&
                     tol > ZERO &&
                     time_end >= time_start &&
                     dt > ZERO &&
                     isfinite((double)dx) &&
                     isfinite((double)dy) &&
                     dx > ZERO &&
                     dy > ZERO);
    printf("LASSI-VERIFY ASSERTIONS [main_scalar_dims_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // arrays for pressure and velocity
  Real* F;
  Real* u;
  Real* G;
  Real* v;

  F = (Real *) calloc (size, sizeof(Real));
  u = (Real *) calloc (size, sizeof(Real));
  G = (Real *) calloc (size, sizeof(Real));
  v = (Real *) calloc (size, sizeof(Real));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Host allocation validity for velocity and intermediate arrays
  {
    bool _t1_pass = (F != NULL && u != NULL && G != NULL && v != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_host_ptrs_flow_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  for (int i = 0; i < size; ++i) {
    F[i] = ZERO;
    u[i] = ZERO;
    G[i] = ZERO;
    v[i] = ZERO;
  }

  // arrays for pressure
  Real* pres_red;
  Real* pres_black;

  pres_red = (Real *) calloc (size_pres, sizeof(Real));
  pres_black = (Real *) calloc (size_pres, sizeof(Real));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Host allocation validity for pressure arrays
  {
    bool _t1_pass = (pres_red != NULL && pres_black != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_host_ptrs_pressure_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  for (int i = 0; i < size_pres; ++i) {
    pres_red[i] = ZERO;
    pres_black[i] = ZERO;
  }

  // print problem size
  printf("Problem size: %d x %d \n", NUM, NUM);

  // block and grid dimensions

  // boundary conditions kernel
  dim3 block_bcs (BLOCK_SIZE, 1);
  dim3 grid_bcs (NUM / BLOCK_SIZE, 1);

  // pressure kernel
  dim3 block_pr (BLOCK_SIZE, 1);
  dim3 grid_pr (NUM / (2 * BLOCK_SIZE), NUM);

  // block and grid dimensions for F
  dim3 block_F (BLOCK_SIZE, 1);
  dim3 grid_F (NUM / BLOCK_SIZE, NUM);

  // block and grid dimensions for G
  dim3 block_G (BLOCK_SIZE, 1);
  dim3 grid_G (NUM / BLOCK_SIZE, NUM);

  // horizontal pressure boundary conditions
  dim3 block_hpbc (BLOCK_SIZE, 1);
  dim3 grid_hpbc (NUM / (2 * BLOCK_SIZE), 1);

  // vertical pressure boundary conditions
  dim3 block_vpbc (BLOCK_SIZE, 1);
  dim3 grid_vpbc (NUM / (2 * BLOCK_SIZE), 1);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Kernel launch configuration validity
  {
    bool _t1_pass = (block_bcs.x > 0 && block_bcs.x <= 1024 &&
                     block_pr.x > 0 && block_pr.x <= 1024 &&
                     block_F.x > 0 && block_F.x <= 1024 &&
                     block_G.x > 0 && block_G.x <= 1024 &&
                     block_hpbc.x > 0 && block_hpbc.x <= 1024 &&
                     block_vpbc.x > 0 && block_vpbc.x <= 1024 &&
                     grid_bcs.x > 0 &&
                     grid_pr.x > 0 && grid_pr.y > 0 &&
                     grid_F.x > 0 && grid_F.y > 0 &&
                     grid_G.x > 0 && grid_G.y > 0 &&
                     grid_hpbc.x > 0 &&
                     grid_vpbc.x > 0);
    printf("LASSI-VERIFY ASSERTIONS [main_kernel_params_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // residual variable
  Real* res;

  int size_res = grid_pr.x * grid_pr.y;
  res = (Real *) calloc (size_res, sizeof(Real));

  // variables to store maximum velocities
  Real* max_u_arr;
  Real* max_v_arr;
  int size_max = grid_pr.x * grid_pr.y;

  max_u_arr = (Real *) calloc (size_max, sizeof(Real));
  max_v_arr = (Real *) calloc (size_max, sizeof(Real));

  // pressure sum
  Real* pres_sum;
  pres_sum = (Real *) calloc (size_res, sizeof(Real));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Host allocation validity for reduction/output helper arrays
  {
    bool _t1_pass = (size_res > 0 &&
                     size_max > 0 &&
                     res != NULL &&
                     max_u_arr != NULL &&
                     max_v_arr != NULL &&
                     pres_sum != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_host_ptrs_aux_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // set initial BCs
  set_BCs_host (u, v);

  Real max_u = SMALL;
  Real max_v = SMALL;
  // get max velocity for initial values (including BCs)
  #pragma unroll
  for (int col = 0; col < NUM + 2; ++col) {
    #pragma unroll
    for (int row = 1; row < NUM + 2; ++row) {
      max_u = fmax(max_u, fabs( u(col, row) ));
    }
  }

  #pragma unroll
  for (int col = 1; col < NUM + 2; ++col) {
    #pragma unroll
    for (int row = 0; row < NUM + 2; ++row) {
      max_v = fmax(max_v, fabs( v(col, row) ));
    }
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Initial host field numerical validity before host-to-device transfer
  {
    bool _t1_pass = true;
    for (int i = 0; i < size; ++i) {
      if (!isfinite((double)F[i]) || !isfinite((double)u[i]) ||
          !isfinite((double)G[i]) || !isfinite((double)v[i])) { _t1_pass = false; break; }
    }
    if (_t1_pass) {
      for (int i = 0; i < size_pres; ++i) {
        if (!isfinite((double)pres_red[i]) || !isfinite((double)pres_black[i])) { _t1_pass = false; break; }
      }
    }
    if (_t1_pass) {
      _t1_pass = (isfinite((double)max_u) && isfinite((double)max_v) && max_u > ZERO && max_v > ZERO);
    }
    printf("LASSI-VERIFY ASSERTIONS [main_input_finite_before_htod]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // allocate and transfer device memory
  Real* u_d;
  Real* F_d;
  Real* v_d;
  Real* G_d;

  Real* pres_red_d;
  Real* pres_black_d;
  Real* pres_sum_d;
  Real* res_d;

  Real* max_u_d;
  Real* max_v_d;

  cudaMalloc ((void**) &u_d, size * sizeof(Real));
  cudaMalloc ((void**) &F_d, size * sizeof(Real));
  cudaMalloc ((void**) &v_d, size * sizeof(Real));
  cudaMalloc ((void**) &G_d, size * sizeof(Real));

  cudaMalloc ((void**) &pres_red_d, size_pres * sizeof(Real));
  cudaMalloc ((void**) &pres_black_d, size_pres * sizeof(Real));

  cudaMalloc ((void**) &pres_sum_d, size_res * sizeof(Real));
  cudaMalloc ((void**) &res_d, size_res * sizeof(Real));
  cudaMalloc ((void**) &max_u_d, size_max * sizeof(Real));
  cudaMalloc ((void**) &max_v_d, size_max * sizeof(Real));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Device allocation pointer validity prior to memory transfers
  {
    bool _t1_pass = (u_d != NULL && F_d != NULL && v_d != NULL && G_d != NULL &&
                     pres_red_d != NULL && pres_black_d != NULL &&
                     pres_sum_d != NULL && res_d != NULL &&
                     max_u_d != NULL && max_v_d != NULL);
    printf("LASSI-VERIFY ASSERTIONS [main_device_ptrs_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // copy to device memory
  cudaMemcpy (u_d, u, size * sizeof(Real), cudaMemcpyHostToDevice);
  cudaMemcpy (F_d, F, size * sizeof(Real), cudaMemcpyHostToDevice);
  cudaMemcpy (v_d, v, size * sizeof(Real), cudaMemcpyHostToDevice);
  cudaMemcpy (G_d, G, size * sizeof(Real), cudaMemcpyHostToDevice);
  cudaMemcpy (pres_red_d, pres_red, size_pres * sizeof(Real), cudaMemcpyHostToDevice);
  cudaMemcpy (pres_black_d, pres_black, size_pres * sizeof(Real), cudaMemcpyHostToDevice);

  Real time = time_start;

  // time-step size based on grid and Reynolds number
  Real dt_Re = 0.5 * Re_num / ((1.0 / (dx * dx)) + (1.0 / (dy * dy)));

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Time-step stability scalar validity before solver loop
  {
    bool _t1_pass = (isfinite((double)time) &&
                     isfinite((double)dt_Re) &&
                     dt_Re > ZERO &&
                     time == time_start);
    printf("LASSI-VERIFY ASSERTIONS [main_preloop_timestep_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaDeviceSynchronize();
  auto start = std::chrono::steady_clock::now();

  // time iteration loop
  while (time < time_end) {

    // calculate time step based on stability and CFL
    dt = fmin((dx / max_u), (dy / max_v));
    dt = tau * fmin(dt_Re, dt);

    if ((time + dt) >= time_end) {
      dt = time_end - time;
    }

    // calculate F and G    
    calculate_F <<<grid_F, block_F>>> (dt, u_d, v_d, F_d);
    calculate_G <<<grid_G, block_G>>> (dt, u_d, v_d, G_d);
    cudaDeviceSynchronize();

    // >>> BEGIN LASSI-VERIFY PASSTHROUGH [post_fg_fields] (do not edit or delete this block) >>>
    {
        // --- LASSI-VERIFY PASSTHROUGH: capture intermediate F field after F/G kernel phase ---
        Real* lv_host_buf = (Real*) malloc(size * sizeof(Real));
        cudaMemcpy(lv_host_buf, F_d, size * sizeof(Real), cudaMemcpyDeviceToHost);

        Real lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
        double lv_sum = 0.0;
        for (int lv_i = 0; lv_i < size; lv_i++) {
            if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
            if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
            lv_sum += (double)lv_host_buf[lv_i];
        }
        double lv_mean = lv_sum / size;

        printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [post_fg_fields] ===\n");
        printf("Buffer: F_d\n");
        printf("Type: Real\n");
        printf("Elements: %d\n", (int)size);
        printf("Statistics: min=%.17g, max=%.17g, mean=%.17g\n",
               (double)lv_min, (double)lv_max, lv_mean);

        int lv_sample = (size < 10) ? (int)size : 10;
        printf("Sample_first10:");
        for (int lv_i = 0; lv_i < lv_sample; lv_i++)
            printf(" %.17g", (double)lv_host_buf[lv_i]);
        printf("\n");

        if (size > 10) {
            printf("Sample_last10:");
            for (int lv_i = (int)size - 10; lv_i < (int)size; lv_i++)
                printf(" %.17g", (double)lv_host_buf[lv_i]);
            printf("\n");
        }
        printf("=== END LASSI-VERIFY PASSTHROUGH [post_fg_fields] ===\n");
        free(lv_host_buf);
    }
    // <<< END LASSI-VERIFY PASSTHROUGH <<<

    // get L2 norm of initial pressure
    sum_pressure <<<grid_pr, block_pr>>> (pres_red_d, pres_black_d, pres_sum_d);
    cudaMemcpy (pres_sum, pres_sum_d, size_res * sizeof(Real), cudaMemcpyDeviceToHost);

    Real p0_norm = ZERO;
    #pragma unroll
    for (int i = 0; i < size_res; ++i) {
      p0_norm += pres_sum[i];
    }

    p0_norm = sqrt(p0_norm / ((Real)(NUM * NUM)));
    if (p0_norm < 0.0001) {
      p0_norm = 1.0;
    }

    // ensure all kernels are finished
    //cudaDeviceSynchronize();

    Real norm_L2;

    // calculate new pressure
    // red-black Gauss-Seidel with SOR iteration loop
    for (iter = 1; iter <= it_max; ++iter) {

      // set pressure boundary conditions
      set_horz_pres_BCs <<<grid_hpbc, block_hpbc>>> (pres_red_d, pres_black_d);
      set_vert_pres_BCs <<<grid_vpbc, block_vpbc>>> (pres_red_d, pres_black_d);

      // ensure kernel finished
      //cudaDeviceSynchronize();

      // update red cells
      red_kernel <<<grid_pr, block_pr>>> (dt, F_d, G_d, pres_black_d, pres_red_d);

      // ensure red kernel finished
      //cudaDeviceSynchronize();

      // update black cells
      black_kernel <<<grid_pr, block_pr>>> (dt, F_d, G_d, pres_red_d, pres_black_d);

      // ensure red kernel finished
      //cudaDeviceSynchronize();

      // calculate residual values
      calc_residual <<<grid_pr, block_pr>>> (dt, F_d, G_d, pres_red_d, pres_black_d, res_d);

      // transfer residual value(s) back to CPU
      cudaMemcpy (res, res_d, size_res * sizeof(Real), cudaMemcpyDeviceToHost);

      norm_L2 = ZERO;
      #pragma unroll
      for (int i = 0; i < size_res; ++i) {
        norm_L2 += res[i];
      }

      // calculate residual
      norm_L2 = sqrt(norm_L2 / ((Real)(NUM * NUM))) / p0_norm;

      // if tolerance has been reached, end SOR iterations
      if (norm_L2 < tol) {
        break;
      }  
    } // end for

    cudaDeviceSynchronize();

    // >>> BEGIN LASSI-VERIFY PASSTHROUGH [post_pressure_solve] (do not edit or delete this block) >>>
    {
        // --- LASSI-VERIFY PASSTHROUGH: capture pressure red field after pressure solve phase ---
        Real* lv_host_buf = (Real*) malloc(size_pres * sizeof(Real));
        cudaMemcpy(lv_host_buf, pres_red_d, size_pres * sizeof(Real), cudaMemcpyDeviceToHost);

        Real lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
        double lv_sum = 0.0;
        for (int lv_i = 0; lv_i < size_pres; lv_i++) {
            if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
            if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
            lv_sum += (double)lv_host_buf[lv_i];
        }
        double lv_mean = lv_sum / size_pres;

        printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [post_pressure_solve] ===\n");
        printf("Buffer: pres_red_d\n");
        printf("Type: Real\n");
        printf("Elements: %d\n", (int)size_pres);
        printf("Statistics: min=%.17g, max=%.17g, mean=%.17g\n",
               (double)lv_min, (double)lv_max, lv_mean);

        int lv_sample = (size_pres < 10) ? (int)size_pres : 10;
        printf("Sample_first10:");
        for (int lv_i = 0; lv_i < lv_sample; lv_i++)
            printf(" %.17g", (double)lv_host_buf[lv_i]);
        printf("\n");

        if (size_pres > 10) {
            printf("Sample_last10:");
            for (int lv_i = (int)size_pres - 10; lv_i < (int)size_pres; lv_i++)
                printf(" %.17g", (double)lv_host_buf[lv_i]);
            printf("\n");
        }
        printf("=== END LASSI-VERIFY PASSTHROUGH [post_pressure_solve] ===\n");
        free(lv_host_buf);
    }
    // <<< END LASSI-VERIFY PASSTHROUGH <<<

    printf("Time = %f, delt = %e, iter = %i, res = %e\n", time + dt, dt, iter, norm_L2);

    // calculate new velocities and transfer maximums back

    calculate_u <<<grid_pr, block_pr>>> (dt, F_d, pres_red_d, pres_black_d, u_d, max_u_d);
    cudaMemcpy (max_u_arr, max_u_d, size_max * sizeof(Real), cudaMemcpyDeviceToHost);

    calculate_v <<<grid_pr, block_pr>>> (dt, G_d, pres_red_d, pres_black_d, v_d, max_v_d);
    cudaMemcpy (max_v_arr, max_v_d, size_max * sizeof(Real), cudaMemcpyDeviceToHost);

    // get maximum u- and v- velocities
    max_v = SMALL;
    max_u = SMALL;

    #pragma unroll
    for (int i = 0; i < size_max; ++i) {
      Real test_u = max_u_arr[i];
      max_u = fmax(max_u, test_u);

      Real test_v = max_v_arr[i];
      max_v = fmax(max_v, test_v);
    }

    // set velocity boundary conditions
    set_BCs <<<grid_bcs, block_bcs>>> (u_d, v_d);
    cudaDeviceSynchronize();

    // >>> BEGIN LASSI-VERIFY PASSTHROUGH [post_timestep_velocity] (do not edit or delete this block) >>>
    {
        // --- LASSI-VERIFY PASSTHROUGH: capture velocity field after timestep update and BC application ---
        Real* lv_host_buf = (Real*) malloc(size * sizeof(Real));
        cudaMemcpy(lv_host_buf, u_d, size * sizeof(Real), cudaMemcpyDeviceToHost);

        Real lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
        double lv_sum = 0.0;
        for (int lv_i = 0; lv_i < size; lv_i++) {
            if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
            if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
            lv_sum += (double)lv_host_buf[lv_i];
        }
        double lv_mean = lv_sum / size;

        printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [post_timestep_velocity] ===\n");
        printf("Buffer: u_d\n");
        printf("Type: Real\n");
        printf("Elements: %d\n", (int)size);
        printf("Statistics: min=%.17g, max=%.17g, mean=%.17g\n",
               (double)lv_min, (double)lv_max, lv_mean);

        int lv_sample = (size < 10) ? (int)size : 10;
        printf("Sample_first10:");
        for (int lv_i = 0; lv_i < lv_sample; lv_i++)
            printf(" %.17g", (double)lv_host_buf[lv_i]);
        printf("\n");

        if (size > 10) {
            printf("Sample_last10:");
            for (int lv_i = (int)size - 10; lv_i < (int)size; lv_i++)
                printf(" %.17g", (double)lv_host_buf[lv_i]);
            printf("\n");
        }
        printf("=== END LASSI-VERIFY PASSTHROUGH [post_timestep_velocity] ===\n");
        free(lv_host_buf);
    }
    // <<< END LASSI-VERIFY PASSTHROUGH <<<

    // increase time
    time += dt;

    // single time step
    //break;

  } // end while

  cudaDeviceSynchronize();
  auto end = std::chrono::steady_clock::now();
  auto elapsed_time = std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count();
  printf("\nTotal execution time of the iteration loop: %f (s)\n", elapsed_time * 1e-9f);

  // transfer final temperature values back
  cudaMemcpy (u, u_d, size * sizeof(Real), cudaMemcpyDeviceToHost);
  cudaMemcpy (v, v_d, size * sizeof(Real), cudaMemcpyDeviceToHost);
  cudaMemcpy (pres_red, pres_red_d, size_pres * sizeof(Real), cudaMemcpyDeviceToHost);
  cudaMemcpy (pres_black, pres_black_d, size_pres * sizeof(Real), cudaMemcpyDeviceToHost);

  // >>> BEGIN LASSI-VERIFY PASSTHROUGH [final_u_field] (do not edit or delete this block) >>>
  {
      // --- LASSI-VERIFY PASSTHROUGH: capture final primary output velocity field from device ---
      Real* lv_host_buf = (Real*) malloc(size * sizeof(Real));
      cudaMemcpy(lv_host_buf, u_d, size * sizeof(Real), cudaMemcpyDeviceToHost);

      Real lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
      double lv_sum = 0.0;
      for (int lv_i = 0; lv_i < size; lv_i++) {
          if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
          if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
          lv_sum += (double)lv_host_buf[lv_i];
      }
      double lv_mean = lv_sum / size;

      printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [final_u_field] ===\n");
      printf("Buffer: u_d\n");
      printf("Type: Real\n");
      printf("Elements: %d\n", (int)size);
      printf("Statistics: min=%.17g, max=%.17g, mean=%.17g\n",
             (double)lv_min, (double)lv_max, lv_mean);

      int lv_sample = (size < 10) ? (int)size : 10;
      printf("Sample_first10:");
      for (int lv_i = 0; lv_i < lv_sample; lv_i++)
          printf(" %.17g", (double)lv_host_buf[lv_i]);
      printf("\n");

      if (size > 10) {
          printf("Sample_last10:");
          for (int lv_i = (int)size - 10; lv_i < (int)size; lv_i++)
              printf(" %.17g", (double)lv_host_buf[lv_i]);
          printf("\n");
      }
      printf("=== END LASSI-VERIFY PASSTHROUGH [final_u_field] ===\n");
      free(lv_host_buf);
  }
  // <<< END LASSI-VERIFY PASSTHROUGH <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Final host arrays finite after device-to-host transfers
  {
    bool _t1_pass = true;
    for (int i = 0; i < size; ++i) {
      if (!isfinite((double)u[i]) || !isfinite((double)v[i])) { _t1_pass = false; break; }
    }
    if (_t1_pass) {
      for (int i = 0; i < size_pres; ++i) {
        if (!isfinite((double)pres_red[i]) || !isfinite((double)pres_black[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_output_finite_after_dtoh]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout

... (31 outputs left out)

Waiting for the system to settle down for 5 seconds...: 100%|██████████| 5/5 [00:05<00:00,  1.00s/it]


executable_file:  /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0314-2026_gpt54_lid-driven-cavity_runtm_NVIDIA__03-15-2026_t17_03_26/lid-driven-cavity-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gpt54_lid-driven-cavity_runtm_NVIDIA_03-15-2026_t17_03_26_Trial1
initial_dir:  /home/mdearing/Research/LASSI/LASSI-Verify
directory_path:  /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0314-2026_gpt54_lid-driven-cavity_runtm_NVIDIA__03-15-2026_t17_03_26
executable:  lid-driven-cavity-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gpt54_lid-driven-cavity_runtm_NVIDIA_03-15-2026_t17_03_26_Trial1
Working directory: /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0314-2026_gpt54_lid-driven-cavity_runtm_NVIDIA__03-15-2026_t17_03_26
Execute command: ['./lid-driven-cavity-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gp

Waiting for the system to settle down for 5 seconds...: 100%|██████████| 5/5 [00:05<00:00,  1.00s/it]


executable_file:  /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0314-2026_gpt54_lid-driven-cavity_runtm_NVIDIA__03-15-2026_t17_03_26/lid-driven-cavity-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gpt54_lid-driven-cavity_runtm_NVIDIA_03-15-2026_t17_03_26_Trial1
initial_dir:  /home/mdearing/Research/LASSI/LASSI-Verify
directory_path:  /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0314-2026_gpt54_lid-driven-cavity_runtm_NVIDIA__03-15-2026_t17_03_26
executable:  lid-driven-cavity-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gpt54_lid-driven-cavity_runtm_NVIDIA_03-15-2026_t17_03_26_Trial1
Working directory: /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0314-2026_gpt54_lid-driven-cavity_runtm_NVIDIA__03-15-2026_t17_03_26
Execute command: ['./lid-driven-cavity-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gp